<a href="https://colab.research.google.com/github/mimansa05/AI-LAB/blob/main/week9_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# 1. AST & LOGICAL EXPRESSION REPRESENTATION
# ==========================================

class Expr:
    pass

class Var(Expr):
    def __init__(self, name):
        self.name = name
    def __repr__(self): return self.name
    def __eq__(self, other): return isinstance(other, Var) and self.name == other.name
    def __hash__(self): return hash(('Var', self.name))

class Const(Expr):
    def __init__(self, name):
        self.name = name
    def __repr__(self): return self.name
    def __eq__(self, other): return isinstance(other, Const) and self.name == other.name
    def __hash__(self): return hash(('Const', self.name))

class Func(Expr):
    def __init__(self, name, args):
        self.name = name
        self.args = args
    def __repr__(self): return f"{self.name}({', '.join(map(str, self.args))})"
    def __eq__(self, other): return isinstance(other, Func) and self.name == other.name and self.args == other.args
    def __hash__(self): return hash(('Func', self.name, tuple(self.args)))

class Pred(Expr):
    def __init__(self, name, args):
        self.name = name
        self.args = args
    def __repr__(self): return f"{self.name}({', '.join(map(str, self.args))})"
    def __eq__(self, other): return isinstance(other, Pred) and self.name == other.name and self.args == other.args
    def __hash__(self): return hash(('Pred', self.name, tuple(self.args)))

class Not(Expr):
    def __init__(self, expr):
        self.expr = expr
    def __repr__(self): return f"¬{self.expr}"
    def __eq__(self, other): return isinstance(other, Not) and self.expr == other.expr
    def __hash__(self): return hash(('Not', self.expr))

class And(Expr):
    def __init__(self, left, right):
        self.left = left
        self.right = right
    def __repr__(self): return f"({self.left} ∧ {self.right})"

class Or(Expr):
    def __init__(self, left, right):
        self.left = left
        self.right = right
    def __repr__(self): return f"({self.left} ∨ {self.right})"

class Implies(Expr):
    def __init__(self, left, right):
        self.left = left
        self.right = right
    def __repr__(self): return f"({self.left} → {self.right})"

class ForAll(Expr):
    def __init__(self, var, expr):
        self.var = var
        self.expr = expr
    def __repr__(self): return f"∀{self.var} {self.expr}"

class Exists(Expr):
    def __init__(self, var, expr):
        self.var = var
        self.expr = expr
    def __repr__(self): return f"∃{self.var} {self.expr}"


# ==========================================
# 2. STEP-BY-STEP CNF CONVERSION PIPELINE
# ==========================================

var_counter = 0
skolem_counter = 0

def fresh_var(base="v"):
    global var_counter
    var_counter += 1
    return Var(f"{base}{var_counter}")

def fresh_skolem(univ_vars):
    global skolem_counter
    skolem_counter += 1
    if not univ_vars:
        return Const(f"SkC{skolem_counter}")
    return Func(f"sk_f{skolem_counter}", list(univ_vars))

def eliminate_implications(expr):
    if isinstance(expr, Implies):
        return Or(Not(eliminate_implications(expr.left)), eliminate_implications(expr.right))
    elif isinstance(expr, Not):
        return Not(eliminate_implications(expr.expr))
    elif isinstance(expr, And):
        return And(eliminate_implications(expr.left), eliminate_implications(expr.right))
    elif isinstance(expr, Or):
        return Or(eliminate_implications(expr.left), eliminate_implications(expr.right))
    elif isinstance(expr, (ForAll, Exists)):
        return type(expr)(expr.var, eliminate_implications(expr.expr))
    return expr

def move_negation_inwards(expr):
    if isinstance(expr, Not):
        inner = expr.expr
        if isinstance(inner, Not):
            return move_negation_inwards(inner.expr)
        elif isinstance(inner, And):
            return Or(move_negation_inwards(Not(inner.left)), move_negation_inwards(Not(inner.right)))
        elif isinstance(inner, Or):
            return And(move_negation_inwards(Not(inner.left)), move_negation_inwards(Not(inner.right)))
        elif isinstance(inner, ForAll):
            return Exists(inner.var, move_negation_inwards(Not(inner.expr)))
        elif isinstance(inner, Exists):
            return ForAll(inner.var, move_negation_inwards(Not(inner.expr)))
        return Not(move_negation_inwards(inner))
    elif isinstance(expr, (And, Or)):
        return type(expr)(move_negation_inwards(expr.left), move_negation_inwards(expr.right))
    elif isinstance(expr, (ForAll, Exists)):
        return type(expr)(expr.var, move_negation_inwards(expr.expr))
    return expr

def substitute(expr, mapping):
    if isinstance(expr, Var):
        return mapping.get(expr.name, expr)
    elif isinstance(expr, (Func, Pred)):
        return type(expr)(expr.name, [substitute(arg, mapping) for arg in expr.args])
    elif isinstance(expr, Not):
        return Not(substitute(expr.expr, mapping))
    elif isinstance(expr, (And, Or)):
        return type(expr)(substitute(expr.left, mapping), substitute(expr.right, mapping))
    elif isinstance(expr, (ForAll, Exists)):
        new_var = mapping.get(expr.var.name, expr.var)
        return type(expr)(new_var, substitute(expr.expr, mapping))
    return expr

def standardize_variables(expr, mapping=None):
    if mapping is None:
        mapping = {}
    if isinstance(expr, (ForAll, Exists)):
        new_v = fresh_var(expr.var.name)
        new_map = mapping.copy()
        new_map[expr.var.name] = new_v
        return type(expr)(new_v, standardize_variables(expr.expr, new_map))
    elif isinstance(expr, (And, Or)):
        return type(expr)(standardize_variables(expr.left, mapping), standardize_variables(expr.right, mapping))
    elif isinstance(expr, Not):
        return Not(standardize_variables(expr.expr, mapping))
    elif isinstance(expr, Pred):
        return Pred(expr.name, [substitute(arg, mapping) for arg in expr.args])
    return expr

def skolemize(expr, univ_vars=None):
    if univ_vars is None:
        univ_vars = []
    if isinstance(expr, ForAll):
        return ForAll(expr.var, skolemize(expr.expr, univ_vars + [expr.var]))
    elif isinstance(expr, Exists):
        sk_term = fresh_skolem(univ_vars)
        sub_expr = substitute(expr.expr, {expr.var.name: sk_term})
        return skolemize(sub_expr, univ_vars)
    elif isinstance(expr, (And, Or)):
        return type(expr)(skolemize(expr.left, univ_vars), skolemize(expr.right, univ_vars))
    elif isinstance(expr, Not):
        return Not(skolemize(expr.expr, univ_vars))
    return expr

def drop_universal(expr):
    if isinstance(expr, ForAll):
        return drop_universal(expr.expr)
    elif isinstance(expr, (And, Or)):
        return type(expr)(drop_universal(expr.left), drop_universal(expr.right))
    elif isinstance(expr, Not):
        return Not(drop_universal(expr.expr))
    return expr

def distribute_or(expr):
    if isinstance(expr, Or):
        A = distribute_or(expr.left)
        B = distribute_or(expr.right)
        if isinstance(A, And):
            return And(distribute_or(Or(A.left, B)), distribute_or(Or(A.right, B)))
        if isinstance(B, And):
            return And(distribute_or(Or(A, B.left)), distribute_or(Or(A, B.right)))
        return Or(A, B)
    elif isinstance(expr, And):
        return And(distribute_or(expr.left), distribute_or(expr.right))
    return expr

def extract_clauses(expr):
    if isinstance(expr, And):
        return extract_clauses(expr.left) + extract_clauses(expr.right)
    elif isinstance(expr, Or):
        return [extract_literals(expr)]
    else:
        return [[expr]]

def extract_literals(expr):
    if isinstance(expr, Or):
        return extract_literals(expr.left) + extract_literals(expr.right)
    return [expr]

def to_cnf_with_steps(formula, label=""):
    print(f"\n--- Converting Formula {label} to CNF ---")
    print(f"Original:        {formula}")
    s1 = eliminate_implications(formula)
    print(f"1. Elim Implies: {s1}")
    s2 = move_negation_inwards(s1)
    print(f"2. Neg Inwards:  {s2}")
    s3 = standardize_variables(s2)
    print(f"3. Standardize:  {s3}")
    s4 = skolemize(s3)
    print(f"4. Skolemize:    {s4}")
    s5 = drop_universal(s4)
    print(f"5. Drop ForAll:  {s5}")
    s6 = distribute_or(s5)
    print(f"6. CNF Form:     {s6}")
    return extract_clauses(s6)


# ==========================================
# 3. UNIFICATION & RESOLUTION ENGINE
# ==========================================

def occur_check(var, term, subst):
    if var == term:
        return True
    elif isinstance(term, Var) and term.name in subst:
        return occur_check(var, subst[term.name], subst)
    elif isinstance(term, Func):
        return any(occur_check(var, arg, subst) for arg in term.args)
    return False

def unify(x, y, subst=None):
    if subst is None:
        subst = {}
    if x == y:
        return subst
    elif isinstance(x, Var):
        return unify_var(x, y, subst)
    elif isinstance(y, Var):
        return unify_var(y, x, subst)
    elif isinstance(x, (Pred, Func)) and isinstance(y, (Pred, Func)):
        if x.name != y.name or len(x.args) != len(y.args):
            return None
        for a, b in zip(x.args, y.args):
            subst = unify(a, b, subst)
            if subst is None:
                return None
        return subst
    return None

def unify_var(var, x, subst):
    if var.name in subst:
        return unify(subst[var.name], x, subst)
    elif isinstance(x, Var) and x.name in subst:
        return unify(var, subst[x.name], subst)
    elif occur_check(var, x, subst):
        return None
    new_subst = subst.copy()
    new_subst[var.name] = x
    return new_subst

def apply_subst_literal(lit, subst):
    if isinstance(lit, Not):
        return Not(Pred(lit.expr.name, [substitute(a, subst) for a in lit.expr.args]))
    return Pred(lit.name, [substitute(a, subst) for a in lit.args])

def standardize_clause(clause):
    mapping = {}
    def rename_term(t):
        if isinstance(t, Var):
            if t.name not in mapping:
                mapping[t.name] = fresh_var("x")
            return mapping[t.name]
        elif isinstance(t, Func):
            return Func(t.name, [rename_term(a) for a in t.args])
        return t

    new_clause = []
    for lit in clause:
        if isinstance(lit, Not):
            new_clause.append(Not(Pred(lit.expr.name, [rename_term(a) for a in lit.expr.args])))
        else:
            new_clause.append(Pred(lit.name, [rename_term(a) for a in lit.args]))
    return new_clause

def resolve_clauses(c1, c2):
    c1 = standardize_clause(c1)
    c2 = standardize_clause(c2)
    resolvents = []
    for i, l1 in enumerate(c1):
        for j, l2 in enumerate(c2):
            if isinstance(l1, Not) and not isinstance(l2, Not) and l1.expr.name == l2.name:
                subst = unify(l1.expr, l2)
            elif not isinstance(l1, Not) and isinstance(l2, Not) and l1.name == l2.expr.name:
                subst = unify(l1, l2.expr)
            else:
                continue
            if subst is not None:
                new_c1 = [apply_subst_literal(lit, subst) for k, lit in enumerate(c1) if k != i]
                new_c2 = [apply_subst_literal(lit, subst) for k, lit in enumerate(c2) if k != j]
                combined = []
                for lit in new_c1 + new_c2:
                    if lit not in combined:
                        combined.append(lit)
                resolvents.append((combined, subst))
    return resolvents

def prove_by_resolution(kb_formulas, goal_formula):
    print("==========================================")
    print("STEP 1: CONVERT KB AND NEGATED GOAL TO CNF")
    print("==========================================")
    clauses = []
    for idx, f in enumerate(kb_formulas):
        clauses.extend(to_cnf_with_steps(f, label=f"KB[{idx+1}]"))
    neg_goal = Not(goal_formula)
    clauses.extend(to_cnf_with_steps(neg_goal, label="Negated Goal"))
    print("\n==========================================")
    print("INITIAL CLAUSE SET (KB ∧ ¬Goal)")
    print("==========================================")
    for idx, c in enumerate(clauses):
        print(f"C{idx+1}: {' ∨ '.join(map(str, c))}")
    print("\n==========================================")
    print("STEP 2: RESOLUTION PROOF TRACE")
    print("==========================================")
    indexed_clauses = [(c, f"Initial C{i+1}") for i, c in enumerate(clauses)]
    seen = {tuple(sorted(map(str, c))) for c in clauses}
    i = 0
    while i < len(indexed_clauses):
        for j in range(i):
            c1, _ = indexed_clauses[i]
            c2, _ = indexed_clauses[j]
            resolvents = resolve_clauses(c1, c2)
            for res, subst in resolvents:
                clause_str = ' ∨ '.join(map(str, res)) if res else "{ } (EMPTY CLAUSE)"
                subst_str = f" with {subst}" if subst else ""
                step_num = len(indexed_clauses) + 1
                if not res:
                    print(f"C{step_num}: {clause_str}  <-- Resolved C{i+1} & C{j+1}{subst_str}")
                    print("\n>>> { } HENCE PROVED! <<<")
                    return True
                sig = tuple(sorted(map(str, res)))
                if sig not in seen:
                    seen.add(sig)
                    print(f"C{step_num}: {clause_str}  <-- Resolved C{i+1} & C{j+1}{subst_str}")
                    indexed_clauses.append((res, f"Resolved C{i+1} & C{j+1}"))
        i += 1
    print("\n>>> COULD NOT PROVE (No empty clause derived) <<<")
    return False

# ==========================================
# 4. EXECUTION: CASE STUDIES FROM IMAGES
# ==========================================

if __name__ == "__main__":
    x, y = Var("x"), Var("y")
    John, Apple, Vegetables, Peanuts, Anil, Harry = (
        Const("John"), Const("Apple"), Const("Vegetables"),
        Const("Peanuts"), Const("Anil"), Const("Harry")
    )
    print("")
    print("CASE STUDY 1: JOHN LIKES PEANUTS")
    print("")
    kb = [
        ForAll(x, Implies(Pred("food", [x]), Pred("likes", [John, x]))),
        And(Pred("food", [Apple]), Pred("food", [Vegetables])),
        ForAll(x, ForAll(y, Implies(And(Pred("eats", [x, y]), Not(Pred("killed", [x]))), Pred("food", [y])))),
        And(Pred("eats", [Anil, Peanuts]), Pred("alive", [Anil])),
        ForAll(x, Implies(Pred("eats", [Anil, x]), Pred("eats", [Harry, x]))),
        ForAll(x, Implies(Not(Pred("killed", [x])), Pred("alive", [x]))),
        ForAll(x, Implies(Pred("alive", [x]), Not(Pred("killed", [x]))))
    ]
    goal = Pred("likes", [John, Peanuts])
    prove_by_resolution(kb, goal)
    print("\n**************************************************")
    print("CASE STUDY 2: Q1 CNF CONVERSION (LAST IMAGE)")
    print("**************************************************")
    q1_formula = Or(
        ForAll(x, Exists(y, And(Pred("Q", [x, y]), Not(Pred("P", [x, y]))))),
        Exists(y, ForAll(x, And(Pred("Q", [x, y]), Not(Pred("R", [x, y])))))
    )
    cnf_clauses = to_cnf_with_steps(q1_formula, label="Q1")
    print("\nFinal CNF Clauses for Q1:")
    for c in cnf_clauses:
        print("  ", " ∨ ".join(map(str, c)))


CASE STUDY 1: JOHN LIKES PEANUTS

STEP 1: CONVERT KB AND NEGATED GOAL TO CNF

--- Converting Formula KB[1] to CNF ---
Original:        ∀x (food(x) → likes(John, x))
1. Elim Implies: ∀x (¬food(x) ∨ likes(John, x))
2. Neg Inwards:  ∀x (¬food(x) ∨ likes(John, x))
3. Standardize:  ∀x1 (¬food(x1) ∨ likes(John, x1))
4. Skolemize:    ∀x1 (¬food(x1) ∨ likes(John, x1))
5. Drop ForAll:  (¬food(x1) ∨ likes(John, x1))
6. CNF Form:     (¬food(x1) ∨ likes(John, x1))

--- Converting Formula KB[2] to CNF ---
Original:        (food(Apple) ∧ food(Vegetables))
1. Elim Implies: (food(Apple) ∧ food(Vegetables))
2. Neg Inwards:  (food(Apple) ∧ food(Vegetables))
3. Standardize:  (food(Apple) ∧ food(Vegetables))
4. Skolemize:    (food(Apple) ∧ food(Vegetables))
5. Drop ForAll:  (food(Apple) ∧ food(Vegetables))
6. CNF Form:     (food(Apple) ∧ food(Vegetables))

--- Converting Formula KB[3] to CNF ---
Original:        ∀x ∀y ((eats(x, y) ∧ ¬killed(x)) → food(y))
1. Elim Implies: ∀x ∀y (¬(eats(x, y) ∧ ¬killed(x)

In [ ]:
# ==========================================
# 1. LOGICAL EXPRESSION CLASSES
# ==========================================

class Expr:
    pass

class Var(Expr):
    def __init__(self, name):
        self.name = name
    def __repr__(self): return self.name
    def __eq__(self, other): return isinstance(other, Var) and self.name == other.name
    def __hash__(self): return hash(('Var', self.name))

class Const(Expr):
    def __init__(self, name):
        self.name = name
    def __repr__(self): return self.name
    def __eq__(self, other): return isinstance(other, Const) and self.name == other.name
    def __hash__(self): return hash(('Const', self.name))

class Pred(Expr):
    def __init__(self, name, args):
        self.name = name
        self.args = args
    def __repr__(self): return f"{self.name}({', '.join(map(str, self.args))})"
    def __eq__(self, other): return isinstance(other, Pred) and self.name == other.name and self.args == other.args
    def __hash__(self): return hash(('Pred', self.name, tuple(self.args)))

class Not(Expr):
    def __init__(self, expr):
        self.expr = expr
    def __repr__(self): return f"¬{self.expr}"
    def __eq__(self, other): return isinstance(other, Not) and self.expr == other.expr
    def __hash__(self): return hash(('Not', self.expr))

class And(Expr):
    def __init__(self, left, right):
        self.left = left
        self.right = right
    def __repr__(self): return f"({self.left} ∧ {self.right})"

class Or(Expr):
    def __init__(self, left, right):
        self.left = left
        self.right = right
    def __repr__(self): return f"({self.left} ∨ {self.right})"

class Implies(Expr):
    def __init__(self, left, right):
        self.left = left
        self.right = right
    def __repr__(self): return f"({self.left} → {self.right})"

class ForAll(Expr):
    def __init__(self, var, expr):
        self.var = var
        self.expr = expr
    def __repr__(self): return f"∀{self.var} {self.expr}"


# ==========================================
# 2. STEP-BY-STEP CNF CONVERSION
# ==========================================

var_counter = 0
def fresh_var(base="v"):
    global var_counter
    var_counter += 1
    return Var(f"{base}{var_counter}")

def eliminate_implications(expr):
    if isinstance(expr, Implies):
        return Or(Not(eliminate_implications(expr.left)), eliminate_implications(expr.right))
    elif isinstance(expr, Not):
        return Not(eliminate_implications(expr.expr))
    elif isinstance(expr, (And, Or)):
        return type(expr)(eliminate_implications(expr.left), eliminate_implications(expr.right))
    elif isinstance(expr, ForAll):
        return ForAll(expr.var, eliminate_implications(expr.expr))
    return expr

def move_negation_inwards(expr):
    if isinstance(expr, Not):
        inner = expr.expr
        if isinstance(inner, Not):
            return move_negation_inwards(inner.expr)
        elif isinstance(inner, And):
            return Or(move_negation_inwards(Not(inner.left)), move_negation_inwards(Not(inner.right)))
        elif isinstance(inner, Or):
            return And(move_negation_inwards(Not(inner.left)), move_negation_inwards(Not(inner.right)))
        return Not(move_negation_inwards(inner))
    elif isinstance(expr, (And, Or)):
        return type(expr)(move_negation_inwards(expr.left), move_negation_inwards(expr.right))
    elif isinstance(expr, ForAll):
        return ForAll(expr.var, move_negation_inwards(expr.expr))
    return expr

def substitute(expr, mapping):
    if isinstance(expr, Var):
        return mapping.get(expr.name, expr)
    elif isinstance(expr, Pred):
        return Pred(expr.name, [substitute(arg, mapping) for arg in expr.args])
    elif isinstance(expr, Not):
        return Not(substitute(expr.expr, mapping))
    elif isinstance(expr, (And, Or)):
        return type(expr)(substitute(expr.left, mapping), substitute(expr.right, mapping))
    elif isinstance(expr, ForAll):
        return ForAll(mapping.get(expr.var.name, expr.var), substitute(expr.expr, mapping))
    return expr

def standardize_variables(expr, mapping=None):
    if mapping is None:
        mapping = {}
    if isinstance(expr, ForAll):
        new_v = fresh_var(expr.var.name)
        new_map = mapping.copy()
        new_map[expr.var.name] = new_v
        return ForAll(new_v, standardize_variables(expr.expr, new_map))
    elif isinstance(expr, (And, Or)):
        return type(expr)(standardize_variables(expr.left, mapping), standardize_variables(expr.right, mapping))
    elif isinstance(expr, Not):
        return Not(standardize_variables(expr.expr, mapping))
    elif isinstance(expr, Pred):
        return Pred(expr.name, [substitute(arg, mapping) for arg in expr.args])
    return expr

def drop_universal(expr):
    if isinstance(expr, ForAll):
        return drop_universal(expr.expr)
    elif isinstance(expr, (And, Or)):
        return type(expr)(drop_universal(expr.left), drop_universal(expr.right))
    elif isinstance(expr, Not):
        return Not(drop_universal(expr.expr))
    return expr

def extract_clauses(expr):
    if isinstance(expr, And):
        return extract_clauses(expr.left) + extract_clauses(expr.right)
    elif isinstance(expr, Or):
        return [extract_literals(expr)]
    return [[expr]]

def extract_literals(expr):
    if isinstance(expr, Or):
        return extract_literals(expr.left) + extract_literals(expr.right)
    return [expr]

def to_cnf_steps(formula, label=""):
    print(f"\n[{label}] Original:        {formula}")
    s1 = eliminate_implications(formula)
    print(f"[{label}] 1. Elim Implies: {s1}")
    s2 = move_negation_inwards(s1)
    print(f"[{label}] 2. Neg Inwards:  {s2}")
    s3 = standardize_variables(s2)
    print(f"[{label}] 3. Standardize:  {s3}")
    s4 = drop_universal(s3)
    print(f"[{label}] 4. Drop ForAll:  {s4}")
    return extract_clauses(s4)


# ==========================================
# 3. UNIFICATION & RESOLUTION ENGINE
# ==========================================

def unify(x, y, subst=None):
    if subst is None:
        subst = {}
    if x == y:
        return subst
    elif isinstance(x, Var):
        return unify_var(x, y, subst)
    elif isinstance(y, Var):
        return unify_var(y, x, subst)
    elif isinstance(x, Pred) and isinstance(y, Pred):
        if x.name != y.name or len(x.args) != len(y.args):
            return None
        for a, b in zip(x.args, y.args):
            subst = unify(a, b, subst)
            if subst is None:
                return None
        return subst
    return None

def unify_var(var, x, subst):
    if var.name in subst:
        return unify(subst[var.name], x, subst)
    elif isinstance(x, Var) and x.name in subst:
        return unify(var, subst[x.name], subst)
    new_subst = subst.copy()
    new_subst[var.name] = x
    return new_subst

def apply_subst_literal(lit, subst):
    if isinstance(lit, Not):
        return Not(Pred(lit.expr.name, [substitute(a, subst) for a in lit.expr.args]))
    return Pred(lit.name, [substitute(a, subst) for a in lit.args])

def resolve_clauses(c1, c2):
    resolvents = []
    for i, l1 in enumerate(c1):
        for j, l2 in enumerate(c2):
            if isinstance(l1, Not) and not isinstance(l2, Not) and l1.expr.name == l2.name:
                subst = unify(l1.expr, l2)
            elif not isinstance(l1, Not) and isinstance(l2, Not) and l1.name == l2.expr.name:
                subst = unify(l1, l2.expr)
            else:
                continue

            if subst is not None:
                new_c1 = [apply_subst_literal(lit, subst) for k, lit in enumerate(c1) if k != i]
                new_c2 = [apply_subst_literal(lit, subst) for k, lit in enumerate(c2) if k != j]
                combined = []
                for lit in new_c1 + new_c2:
                    if lit not in combined:
                        combined.append(lit)
                resolvents.append((combined, subst))
    return resolvents

def prove_john_likes_peanuts(kb_formulas, goal_formula):
    print("==========================================")
    print("STEP 1: CONVERT KB & NEGATED GOAL TO CNF")
    print("==========================================")
    clauses = []
    for idx, f in enumerate(kb_formulas):
        clauses.extend(to_cnf_steps(f, label=f"Premise {idx+1}"))

    clauses.extend(to_cnf_steps(Not(goal_formula), label="Negated Goal"))

    print("\n==========================================")
    print("INITIAL CLAUSE SET")
    print("==========================================")
    for idx, c in enumerate(clauses):
        print(f"C{idx+1}: {' ∨ '.join(map(str, c))}")

    print("\n==========================================")
    print("STEP 2: RESOLUTION PROOF TRACE")
    print("==========================================")
    indexed_clauses = list(clauses)
    seen = {tuple(sorted(map(str, c))) for c in clauses}

    i = 0
    while i < len(indexed_clauses):
        for j in range(i):
            resolvents = resolve_clauses(indexed_clauses[i], indexed_clauses[j])
            for res, subst in resolvents:
                clause_str = ' ∨ '.join(map(str, res)) if res else "{ } (EMPTY CLAUSE)"
                subst_str = f" using {subst}" if subst else ""
                step_num = len(indexed_clauses) + 1

                if not res:
                    print(f"C{step_num}: {clause_str}  <-- Resolved C{i+1} & C{j+1}{subst_str}")
                    print("\n>>> { } HENCE PROVED: John likes peanuts! <<<")
                    return True

                sig = tuple(sorted(map(str, res)))
                if sig not in seen:
                    seen.add(sig)
                    print(f"C{step_num}: {clause_str}  <-- Resolved C{i+1} & C{j+1}{subst_str}")
                    indexed_clauses.append(res)
        i += 1
    return False


# ==========================================
# 4. RUN "JOHN LIKES PEANUTS"


In [ ]:
import re
import matplotlib.pyplot as plt

# ==========================================
# 1. GENERIC AST CLASSES
# ==========================================

class Expr:
    pass

class Var(Expr):
    def __init__(self, name): self.name = name
    def __repr__(self): return self.name
    def __eq__(self, o): return isinstance(o, Var) and self.name == o.name
    def __hash__(self): return hash(('Var', self.name))

class Const(Expr):
    def __init__(self, name): self.name = name
    def __repr__(self): return self.name
    def __eq__(self, o): return isinstance(o, Const) and self.name == o.name
    def __hash__(self): return hash(('Const', self.name))

class Func(Expr):
    def __init__(self, name, args): self.name, self.args = name, args
    def __repr__(self): return f"{self.name}({', '.join(map(str, self.args))})"
    def __eq__(self, o): return isinstance(o, Func) and self.name == o.name and self.args == o.args
    def __hash__(self): return hash(('Func', self.name, tuple(self.args)))

class Pred(Expr):
    def __init__(self, name, args): self.name, self.args = name, args
    def __repr__(self): return f"{self.name}({', '.join(map(str, self.args))})"
    def __eq__(self, o): return isinstance(o, Pred) and self.name == o.name and self.args == o.args
    def __hash__(self): return hash(('Pred', self.name, tuple(self.args)))

class Not(Expr):
    def __init__(self, expr): self.expr = expr
    def __repr__(self): return f"¬{self.expr}"
    def __eq__(self, o): return isinstance(o, Not) and self.expr == o.expr
    def __hash__(self): return hash(('Not', self.expr))

class And(Expr):
    def __init__(self, left, right): self.left, self.right = left, right
    def __repr__(self): return f"({self.left} ∧ {self.right})"

class Or(Expr):
    def __init__(self, left, right): self.left, self.right = left, right
    def __repr__(self): return f"({self.left} ∨ {self.right})"

class Implies(Expr):
    def __init__(self, left, right): self.left, self.right = left, right
    def __repr__(self): return f"({self.left} → {self.right})"

class ForAll(Expr):
    def __init__(self, var, expr): self.var, self.expr = var, expr
    def __repr__(self): return f"∀{self.var} {self.expr}"

class Exists(Expr):
    def __init__(self, var, expr): self.var, self.expr = var, expr
    def __repr__(self): return f"∃{self.var} {self.expr}"


# ==========================================
# 2. GENERIC FOL STRING PARSER
# ==========================================

TOKEN_REGEX = re.compile(
    r'\s*(?:(forall|exists|∀|∃|->|→|<->|↔|~|¬|&|∧|\||∨|\(|\)|\[|\]|,|:)|([A-Za-z_][A-Za-z0-9_]*))'
)

def tokenize(text):
    tokens = []
    pos = 0
    while pos < len(text):
        m = TOKEN_REGEX.match(text, pos)
        if not m:
            raise SyntaxError(f"Unexpected character at position {pos}: {text[pos:]}")
        pos = m.end()
        tok = m.group(1) or m.group(2)
        if tok != ':':  # Ignore optional colons like '∀x:'
            tokens.append(tok)
    return tokens

class FOLParser:
    def __init__(self, tokens):
        self.tokens = tokens
        self.pos = 0

    def peek(self, offset=0):
        idx = self.pos + offset
        return self.tokens[idx] if idx < len(self.tokens) else None

    def consume(self, expected=None):
        tok = self.peek()
        if expected and tok != expected:
            raise SyntaxError(f"Expected '{expected}', got '{tok}'")
        self.pos += 1
        return tok

    def parse(self, bound_vars=None):
        if bound_vars is None:
            bound_vars = set()
        return self.parse_iff(bound_vars)

    def parse_iff(self, bound):
        left = self.parse_implies(bound)
        while self.peek() in ('<->', '↔'):
            self.consume()
            right = self.parse_implies(bound)
            left = And(Implies(left, right), Implies(right, left))
        return left

    def parse_implies(self, bound):
        left = self.parse_or(bound)
        if self.peek() in ('->', '→'):
            self.consume()
            right = self.parse_implies(bound)
            return Implies(left, right)
        return left

    def parse_or(self, bound):
        left = self.parse_and(bound)
        while self.peek() in ('|', '∨'):
            self.consume()
            right = self.parse_and(bound)
            left = Or(left, right)
        return left

    def parse_and(self, bound):
        left = self.parse_unary(bound)
        while self.peek() in ('&', '∧'):
            self.consume()
            right = self.parse_unary(bound)
            left = And(left, right)
        return left

    def parse_unary(self, bound):
        tok = self.peek()
        if tok in ('~', '¬'):
            self.consume()
            return Not(self.parse_unary(bound))

        # Handle parenthesized quantifier e.g. (∀x)(∃y)[...]
        if tok in ('(', '[') and self.peek(1) in ('forall', 'exists', '∀', '∃'):
            close_bracket = ')' if tok == '(' else ']'
            self.consume()
            q = self.consume()
            var_name = self.consume()
            self.consume(close_bracket)
            body = self.parse_unary(bound | {var_name})
            return ForAll(Var(var_name), body) if q in ('forall', '∀') else Exists(Var(var_name), body)

        # Handle inline quantifier e.g. ∀x ∀y P(x,y)
        if tok in ('forall', 'exists', '∀', '∃'):
            q = self.consume()
            var_name = self.consume()
            body = self.parse_unary(bound | {var_name})
            return ForAll(Var(var_name), body) if q in ('forall', '∀') else Exists(Var(var_name), body)

        if tok in ('(', '['):
            close_bracket = ')' if tok == '(' else ']'
            self.consume()
            expr = self.parse_iff(bound)
            self.consume(close_bracket)
            return expr

        return self.parse_predicate(bound)

    def parse_predicate(self, bound):
        name = self.consume()
        args = []
        if self.peek() == '(':
            self.consume('(')
            if self.peek() != ')':
                args.append(self.parse_term(bound))
                while self.peek() == ',':
                    self.consume(',')
                    args.append(self.parse_term(bound))
            self.consume(')')
        return Pred(name, args)

    def parse_term(self, bound):
        name = self.consume()
        if self.peek() == '(':
            self.consume('(')
            args = []
            if self.peek() != ')':
                args.append(self.parse_term(bound))
                while self.peek() == ',':
                    self.consume(',')
                    args.append(self.parse_term(bound))
            self.consume(')')
            return Func(name, args)
        return Var(name) if name in bound else Const(name)

def parse_fol(formula_str):
    return FOLParser(tokenize(formula_str)).parse()


# ==========================================
# 3. STEP-BY-STEP CNF CONVERSION ENGINE
# ==========================================

var_counter = 0
skolem_counter = 0

def fresh_var(base="v"):
    global var_counter
    var_counter += 1
    return Var(f"{base}{var_counter}")

def fresh_skolem(univ_vars):
    global skolem_counter
    skolem_counter += 1
    if not univ_vars:
        return Const(f"SkC{skolem_counter}")
    return Func(f"f{skolem_counter}", list(univ_vars))

def eliminate_implications(expr):
    if isinstance(expr, Implies):
        return Or(Not(eliminate_implications(expr.left)), eliminate_implications(expr.right))
    elif isinstance(expr, Not):
        return Not(eliminate_implications(expr.expr))
    elif isinstance(expr, (And, Or)):
        return type(expr)(eliminate_implications(expr.left), eliminate_implications(expr.right))
    elif isinstance(expr, (ForAll, Exists)):
        return type(expr)(expr.var, eliminate_implications(expr.expr))
    return expr

def move_negation_inwards(expr):
    if isinstance(expr, Not):
        inner = expr.expr
        if isinstance(inner, Not):
            return move_negation_inwards(inner.expr)
        elif isinstance(inner, And):
            return Or(move_negation_inwards(Not(inner.left)), move_negation_inwards(Not(inner.right)))
        elif isinstance(inner, Or):
            return And(move_negation_inwards(Not(inner.left)), move_negation_inwards(Not(inner.right)))
        elif isinstance(inner, ForAll):
            return Exists(inner.var, move_negation_inwards(Not(inner.expr)))
        elif isinstance(inner, Exists):
            return ForAll(inner.var, move_negation_inwards(Not(inner.expr)))
        return Not(move_negation_inwards(inner))
    elif isinstance(expr, (And, Or)):
        return type(expr)(move_negation_inwards(expr.left), move_negation_inwards(expr.right))
    elif isinstance(expr, (ForAll, Exists)):
        return type(expr)(expr.var, move_negation_inwards(expr.expr))
    return expr

def substitute(expr, mapping):
    if isinstance(expr, Var):
        return mapping.get(expr.name, expr)
    elif isinstance(expr, (Func, Pred)):
        return type(expr)(expr.name, [substitute(arg, mapping) for arg in expr.args])
    elif isinstance(expr, Not):
        return Not(substitute(expr.expr, mapping))
    elif isinstance(expr, (And, Or)):
        return type(expr)(substitute(expr.left, mapping), substitute(expr.right, mapping))
    elif isinstance(expr, (ForAll, Exists)):
        return type(expr)(mapping.get(expr.var.name, expr.var), substitute(expr.expr, mapping))
    return expr

def standardize_variables(expr, mapping=None):
    if mapping is None:
        mapping = {}
    if isinstance(expr, (ForAll, Exists)):
        new_v = fresh_var(expr.var.name)
        new_map = mapping.copy()
        new_map[expr.var.name] = new_v
        return type(expr)(new_v, standardize_variables(expr.expr, new_map))
    elif isinstance(expr, (And, Or)):
        return type(expr)(standardize_variables(expr.left, mapping), standardize_variables(expr.right, mapping))
    elif isinstance(expr, Not):
        return Not(standardize_variables(expr.expr, mapping))
    elif isinstance(expr, Pred):
        return Pred(expr.name, [substitute(arg, mapping) for arg in expr.args])
    return expr

def skolemize(expr, univ_vars=None):
    if univ_vars is None:
        univ_vars = []
    if isinstance(expr, ForAll):
        return ForAll(expr.var, skolemize(expr.expr, univ_vars + [expr.var]))
    elif isinstance(expr, Exists):
        sk_term = fresh_skolem(univ_vars)
        return skolemize(substitute(expr.expr, {expr.var.name: sk_term}), univ_vars)
    elif isinstance(expr, (And, Or)):
        return type(expr)(skolemize(expr.left, univ_vars), skolemize(expr.right, univ_vars))
    elif isinstance(expr, Not):
        return Not(skolemize(expr.expr, univ_vars))
    return expr

def drop_universal(expr):
    if isinstance(expr, ForAll):
        return drop_universal(expr.expr)
    elif isinstance(expr, (And, Or)):
        return type(expr)(drop_universal(expr.left), drop_universal(expr.right))
    elif isinstance(expr, Not):
        return Not(drop_universal(expr.expr))
    return expr

def distribute_or(expr):
    if isinstance(expr, Or):
        A = distribute_or(expr.left)
        B = distribute_or(expr.right)
        if isinstance(A, And):
            return And(distribute_or(Or(A.left, B)), distribute_or(Or(A.right, B)))
        if isinstance(B, And):
            return And(distribute_or(Or(A, B.left)), distribute_or(Or(A, B.right)))
        return Or(A, B)
    elif isinstance(expr, And):
        return And(distribute_or(expr.left), distribute_or(expr.right))
    return expr

def extract_clauses(expr):
    if isinstance(expr, And):
        return extract_clauses(expr.left) + extract_clauses(expr.right)
    elif isinstance(expr, Or):
        return [extract_literals(expr)]
    return [[expr]]

def extract_literals(expr):
    if isinstance(expr, Or):
        return extract_literals(expr.left) + extract_literals(expr.right)
    return [expr]

def convert_to_cnf(formula_str, label="", verbose=True):
    expr = parse_fol(formula_str)
    s1 = eliminate_implications(expr)
    s2 = move_negation_inwards(s1)
    s3 = standardize_variables(s2)
    s4 = skolemize(s3)
    s5 = drop_universal(s4)
    s6 = distribute_or(s5)
    if verbose:
        print(f"\n--- {label} ---")
        print(f"Parsed FOL:      {expr}")
        print(f"1. Elim Implies: {s1}")
        print(f"2. Neg Inwards:  {s2}")
        print(f"3. Standardize:  {s3}")
        print(f"4. Skolemize:    {s4}")
        print(f"5. Drop ForAll:  {s5}")
        print(f"6. CNF Form:     {s6}")
    return extract_clauses(s6)

# ==========================================
# 4. UNIFICATION & RESOLUTION ENGINE
# ==========================================

def occur_check(var, term, subst):
    if var == term: return True
    if isinstance(term, Var) and term.name in subst:
        return occur_check(var, subst[term.name], subst)
    if isinstance(term, Func):
        return any(occur_check(var, arg, subst) for arg in term.args)
    return False

def unify(x, y, subst=None):
    if subst is None: subst = {}
    if x == y: return subst
    if isinstance(x, Var): return unify_var(x, y, subst)
    if isinstance(y, Var): return unify_var(y, x, subst)
    if isinstance(x, (Pred, Func)) and isinstance(y, (Pred, Func)):
        if x.name != y.name or len(x.args) != len(y.args): return None
        for a, b in zip(x.args, y.args):
            subst = unify(a, b, subst)
            if subst is None: return None
        return subst
    return None

def unify_var(var, x, subst):
    if var.name in subst: return unify(subst[var.name], x, subst)
    if isinstance(x, Var) and x.name in subst: return unify(var, subst[x.name], subst)
    if occur_check(var, x, subst): return None
    new_subst = subst.copy()
    new_subst[var.name] = x
    return new_subst

def apply_subst_literal(lit, subst):
    if isinstance(lit, Not):
        return Not(Pred(lit.expr.name, [substitute(a, subst) for a in lit.expr.args]))
    return Pred(lit.name, [substitute(a, subst) for a in lit.args])

def standardize_clause(clause):
    mapping = {}
    def rename(t):
        if isinstance(t, Var):
            if t.name not in mapping: mapping[t.name] = fresh_var("v")
            return mapping[t.name]
        if isinstance(t, Func):
            return Func(t.name, [rename(a) for a in t.args])
        return t
    res = []
    for lit in clause:
        if isinstance(lit, Not):
            res.append(Not(Pred(lit.expr.name, [rename(a) for a in lit.expr.args])))
        else:
            res.append(Pred(lit.name, [rename(a) for a in lit.args]))
    return res

def resolve_clauses(c1, c2):
    c1_std = standardize_clause(c1)
    c2_std = standardize_clause(c2)
    resolvents = []
    for i, l1 in enumerate(c1_std):
        for j, l2 in enumerate(c2_std):
            if isinstance(l1, Not) and not isinstance(l2, Not) and l1.expr.name == l2.name:
                subst = unify(l1.expr, l2)
            elif not isinstance(l1, Not) and isinstance(l2, Not) and l1.name == l2.expr.name:
                subst = unify(l1, l2.expr)
            else:
                continue
            if subst is not None:
                rem1 = [apply_subst_literal(lit, subst) for k, lit in enumerate(c1_std) if k != i]
                rem2 = [apply_subst_literal(lit, subst) for k, lit in enumerate(c2_std) if k != j]
                combined = []
                for lit in rem1 + rem2:
                    if lit not in combined:
                        combined.append(lit)
                resolvents.append((combined, subst))
    return resolvents

# ==========================================
# 5. DYNAMIC PROOF TREE VISUALIZER
# ==========================================

def plot_dynamic_proof_tree(nodes, goal_idx, title="Dynamic Resolution Proof Tree"):
    children_map = {}
    depths = {}
    def compute_depth(idx):
        if idx in depths:
            return depths[idx]
        _, p1, p2, _ = nodes[idx]
        if p1 is None:
            depths[idx] = 0
        else:
            children_map[idx] = (p1, p2)
            depths[idx] = 1 + max(compute_depth(p1), compute_depth(p2))
        return depths[idx]
    max_depth = compute_depth(goal_idx)

    coords = {}
    leaf_counter = [0]
    def assign_coords(idx):
        _, p1, p2, _ = nodes[idx]
        y = max_depth - depths[idx]
        if p1 is None:
            x = leaf_counter[0] * 3.2
            leaf_counter[0] += 1
        else:
            assign_coords(p1)
            assign_coords(p2)
            x = (coords[p1][0] + coords[p2][0]) / 2.0
        coords[idx] = (x, y)
    assign_coords(goal_idx)

    fig_w = max(9, leaf_counter[0] * 2.6)
    fig_h = max(6, (max_depth + 1) * 1.6)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=140)
    ax.axis('off')
    ax.set_title(title, fontsize=14, fontweight='bold', pad=15)
    for idx, (x, y) in coords.items():
        clause, p1, p2, subst = nodes[idx]
        label = " ∨ ".join(map(str, clause)) if clause else "{ }  Hence Proved."
        if not clause:
            fc, ec, fw = '#f0fff4', '#2f855a', 'bold'
        elif p1 is None:
            fc, ec, fw = '#fffaf0', '#dd6b20', 'normal'
        else:
            fc, ec, fw = '#ebf8ff', '#2b6cb0', 'normal'
        ax.text(x, y, label, ha='center', va='center', fontsize=9.5, fontweight=fw,
                bbox=dict(boxstyle="round,pad=0.4", fc=fc, ec=ec, lw=1.5), zorder=3)
        if p1 is not None:
            x1, y1 = coords[p1]
            x2, y2 = coords[p2]
            ax.plot([x1, x], [y1 - 0.15, y + 0.15], color='#4a5568', lw=1.3, zorder=1)
            ax.plot([x2, x], [y2 - 0.15, y + 0.15], color='#4a5568', lw=1.3, zorder=1)
            if subst:
                subst_txt = "{" + ", ".join(f"{v}/{k}" for k, v in subst.items()) + "}"
                ax.text(x + 0.25, (y1 + y) / 2.0, subst_txt, fontsize=8.5, color='#c53030', fontweight='bold')
    xs = [c[0] for c in coords.values()]
    ys = [c[1] for c in coords.values()]
    ax.set_xlim(min(xs) - 2.2, max(xs) + 2.2)
    ax.set_ylim(min(ys) - 0.6, max(ys) + 0.6)
    plt.tight_layout()
    plt.show()

# ==========================================
# 6. GENERIC RESOLUTION SOLVER DRIVER
# ==========================================

def solve_resolution(kb_strings, goal_string, verbose_cnf=True):
    global var_counter, skolem_counter
    var_counter = 0
    skolem_counter = 0
    print("==========================================")
    print("STEP 1: PARSE & CONVERT TO CNF")
    print("==========================================")
    nodes = []
    for i, s in enumerate(kb_strings):
        for c in convert_to_cnf(s, label=f"KB[{i+1}]", verbose=verbose_cnf):
            nodes.append((c, None, None, None))
    neg_goal_str = f"~({goal_string})"
    for c in convert_to_cnf(neg_goal_str, label="Negated Goal", verbose=verbose_cnf):
        nodes.append((c, None, None, None))
    print("\n==========================================")
    print("INITIAL CLAUSE SET")
    print("==========================================")
    for idx, (c, _, _, _) in enumerate(nodes):
        print(f"C{idx+1}: {' ∨ '.join(map(str, c))}")
    print("\n==========================================")
    print("STEP 2: RESOLUTION TRACE")
    print("==========================================")
    seen = {tuple(sorted(map(str, c))) for c, _, _, _ in nodes}
    i = 0
    while i < len(nodes):
        for j in range(i):
            resolvents = resolve_clauses(nodes[i][0], nodes[j][0])
            for res, subst in resolvents:
                step_num = len(nodes) + 1
                subst_str = f" using {subst}" if subst else ""
                if not res:
                    print(f"C{step_num}: {{ }} (EMPTY CLAUSE) <-- Resolved C{i+1} & C{j+1}{subst_str}")
                    print("\n>>> { } HENCE PROVED! <<<")
                    nodes.append(([], i, j, subst))
                    plot_dynamic_proof_tree(nodes, len(nodes) - 1, title=f"Proof Tree: {goal_string}")
                    return True
                sig = tuple(sorted(map(str, res)))
                if sig not in seen:
                    seen.add(sig)
                    print(f"C{step_num}: {' ∨ '.join(map(str, res))} <-- Resolved C{i+1} & C{j+1}{subst_str}")
                    nodes.append((res, i, j, subst))
        i += 1
    print("\n>>> COULD NOT PROVE (Empty clause {} cannot be derived) <<<")
    return False

# ==========================================
# 7. GENERIC INPUT RUNNER
# ==========================================

if __name__ == "__main__":
    kb_input = [
        "∀x: food(x) -> likes(John, x)",
        "food(Apple) ∧ food(vegetables)",
        "∀x ∀y: (eats(x, y) ∧ ¬killed(x)) -> food(y)",
        "eats(Anil, Peanuts) ∧ alive(Anil)",
        "∀x: eats(Anil, x) -> eats(Harry, x)",
        "∀x: ¬killed(x) -> alive(x)",
        "∀x: alive(x) -> ¬killed(x)"
    ]
    goal_input = "likes(John, Peanuts)"
    solve_resolution(kb_input, goal_input, verbose_cnf=True)

STEP 1: PARSE & CONVERT TO CNF

--- KB[1] ---
Parsed FOL:      (∀x food(x) → likes(John, x))
1. Elim Implies: (¬∀x food(x) ∨ likes(John, x))
2. Neg Inwards:  (∃x ¬food(x) ∨ likes(John, x))
3. Standardize:  (∃x1 ¬food(x1) ∨ likes(John, x))
4. Skolemize:    (¬food(SkC1) ∨ likes(John, x))
5. Drop ForAll:  (¬food(SkC1) ∨ likes(John, x))
6. CNF Form:     (¬food(SkC1) ∨ likes(John, x))

--- KB[2] ---
Parsed FOL:      (food(Apple) ∧ food(vegetables))
1. Elim Implies: (food(Apple) ∧ food(vegetables))
2. Neg Inwards:  (food(Apple) ∧ food(vegetables))
3. Standardize:  (food(Apple) ∧ food(vegetables))
4. Skolemize:    (food(Apple) ∧ food(vegetables))
5. Drop ForAll:  (food(Apple) ∧ food(vegetables))
6. CNF Form:     (food(Apple) ∧ food(vegetables))

--- KB[3] ---
Parsed FOL:      (∀x ∀y (eats(x, y) ∧ ¬killed(x)) → food(y))
1. Elim Implies: (¬∀x ∀y (eats(x, y) ∧ ¬killed(x)) ∨ food(y))
2. Neg Inwards:  (∃x ∃y (¬eats(x, y) ∨ killed(x)) ∨ food(y))
3. Standardize:  (∃x2 ∃y3 (¬eats(x2, y3) ∨ killed(x2)

In [ ]:
import re
import matplotlib.pyplot as plt

# ==========================================
# 1. GENERIC AST CLASSES
# ==========================================

class Expr:
    pass

class Var(Expr):
    def __init__(self, name): self.name = name
    def __repr__(self): return self.name
    def __eq__(self, o): return isinstance(o, Var) and self.name == o.name
    def __hash__(self): return hash(('Var', self.name))

class Const(Expr):
    def __init__(self, name): self.name = name
    def __repr__(self): return self.name
    def __eq__(self, o): return isinstance(o, Const) and self.name == o.name
    def __hash__(self): return hash(('Const', self.name))

class Func(Expr):
    def __init__(self, name, args): self.name, self.args = name, args
    def __repr__(self): return f"{self.name}({', '.join(map(str, self.args))})"
    def __eq__(self, o): return isinstance(o, Func) and self.name == o.name and self.args == o.args
    def __hash__(self): return hash(('Func', self.name, tuple(self.args)))

class Pred(Expr):
    def __init__(self, name, args): self.name, self.args = name, args
    def __repr__(self): return f"{self.name}({', '.join(map(str, self.args))})"
    def __eq__(self, o): return isinstance(o, Pred) and self.name == o.name and self.args == o.args
    def __hash__(self): return hash(('Pred', self.name, tuple(self.args)))

class Not(Expr):
    def __init__(self, expr): self.expr = expr
    def __repr__(self): return f"¬{self.expr}"
    def __eq__(self, o): return isinstance(o, Not) and self.expr == o.expr
    def __hash__(self): return hash(('Not', self.expr))

class And(Expr):
    def __init__(self, left, right): self.left, self.right = left, right
    def __repr__(self): return f"({self.left} ∧ {self.right})"

class Or(Expr):
    def __init__(self, left, right): self.left, self.right = left, right
    def __repr__(self): return f"({self.left} ∨ {self.right})"

class Implies(Expr):
    def __init__(self, left, right): self.left, self.right = left, right
    def __repr__(self): return f"({self.left} → {self.right})"

class ForAll(Expr):
    def __init__(self, var, expr): self.var, self.expr = var, expr
    def __repr__(self): return f"∀{self.var} {self.expr}"

class Exists(Expr):
    def __init__(self, var, expr): self.var, self.expr = var, expr
    def __repr__(self): return f"∃{self.var} {self.expr}"


# ==========================================
# 2. GENERIC FOL STRING PARSER
# ==========================================

TOKEN_REGEX = re.compile(
    r'\s*(?:(forall|exists|∀|∃|->|→|<->|↔|~|¬|&|∧|\||∨|\(|\)|\[|\]|,|:)|([A-Za-z_][A-Za-z0-9_]*))'
)

def tokenize(text):
    tokens = []
    pos = 0
    while pos < len(text):
        m = TOKEN_REGEX.match(text, pos)
        if not m:
            raise SyntaxError(f"Unexpected character at position {pos}: {text[pos:]}")
        pos = m.end()
        tok = m.group(1) or m.group(2)
        if tok != ':':
            tokens.append(tok)
    return tokens

class FOLParser:
    def __init__(self, tokens):
        self.tokens = tokens
        self.pos = 0

    def peek(self, offset=0):
        idx = self.pos + offset
        return self.tokens[idx] if idx < len(self.tokens) else None

    def consume(self, expected=None):
        tok = self.peek()
        if expected and tok != expected:
            raise SyntaxError(f"Expected '{expected}', got '{tok}'")
        self.pos += 1
        return tok

    def parse(self, bound_vars=None):
        if bound_vars is None:
            bound_vars = set()
        return self.parse_quantified(bound_vars)

    def parse_quantified(self, bound):
        tok = self.peek()
        if tok in ('(', '[') and self.peek(1) in ('forall', 'exists', '∀', '∃'):
            close_bracket = ')' if tok == '(' else ']'
            self.consume()
            q = self.consume()
            var_name = self.consume()
            self.consume(close_bracket)
            body = self.parse_quantified(bound | {var_name})
            return ForAll(Var(var_name), body) if q in ('forall', '∀') else Exists(Var(var_name), body)

        if tok in ('forall', 'exists', '∀', '∃'):
            q = self.consume()
            var_name = self.consume()
            body = self.parse_quantified(bound | {var_name})
            return ForAll(Var(var_name), body) if q in ('forall', '∀') else Exists(Var(var_name), body)

        return self.parse_iff(bound)

    def parse_iff(self, bound):
        left = self.parse_implies(bound)
        while self.peek() in ('<->', '↔'):
            self.consume()
            right = self.parse_implies(bound)
            left = And(Implies(left, right), Implies(right, left))
        return left

    def parse_implies(self, bound):
        left = self.parse_or(bound)
        if self.peek() in ('->', '→'):
            self.consume()
            right = self.parse_implies(bound)
            return Implies(left, right)
        return left

    def parse_or(self, bound):
        left = self.parse_and(bound)
        while self.peek() in ('|', '∨'):
            self.consume()
            right = self.parse_and(bound)
            left = Or(left, right)
        return left

    def parse_and(self, bound):
        left = self.parse_unary(bound)
        while self.peek() in ('&', '∧'):
            self.consume()
            right = self.parse_unary(bound)
            left = And(left, right)
        return left

    def parse_unary(self, bound):
        tok = self.peek()
        if tok in ('~', '¬'):
            self.consume()
            return Not(self.parse_unary(bound))

        if tok in ('(', '['):
            close_bracket = ')' if tok == '(' else ']'
            self.consume()
            expr = self.parse_quantified(bound)
            self.consume(close_bracket)
            return expr

        return self.parse_predicate(bound)

    def parse_predicate(self, bound):
        name = self.consume()
        args = []
        if self.peek() == '(':
            self.consume('(')
            if self.peek() != ')':
                args.append(self.parse_term(bound))
                while self.peek() == ',':
                    self.consume(',')
                    args.append(self.parse_term(bound))
            self.consume(')')
        return Pred(name, args)

    def parse_term(self, bound):
        name = self.consume()
        if self.peek() == '(':
            self.consume('(')
            args = []
            if self.peek() != ')':
                args.append(self.parse_term(bound))
                while self.peek() == ',':
                    self.consume(',')
                    args.append(self.parse_term(bound))
            self.consume(')')
            return Func(name, args)
        return Var(name) if name in bound else Const(name)

def parse_fol(formula_str):
    return FOLParser(tokenize(formula_str)).parse()


# ==========================================
# 3. STEP-BY-STEP CNF CONVERSION ENGINE
# ==========================================

var_counter = 0
skolem_counter = 0

def fresh_var(base="v"):
    global var_counter
    var_counter += 1
    return Var(f"{base}{var_counter}")

def fresh_skolem(univ_vars):
    global skolem_counter
    skolem_counter += 1
    if not univ_vars:
        return Const(f"SkC{skolem_counter}")
    return Func(f"f{skolem_counter}", list(univ_vars))

def eliminate_implications(expr):
    if isinstance(expr, Implies):
        return Or(Not(eliminate_implications(expr.left)), eliminate_implications(expr.right))
    elif isinstance(expr, Not):
        return Not(eliminate_implications(expr.expr))
    elif isinstance(expr, (And, Or)):
        return type(expr)(eliminate_implications(expr.left), eliminate_implications(expr.right))
    elif isinstance(expr, (ForAll, Exists)):
        return type(expr)(expr.var, eliminate_implications(expr.expr))
    return expr

def move_negation_inwards(expr):
    if isinstance(expr, Not):
        inner = expr.expr
        if isinstance(inner, Not):
            return move_negation_inwards(inner.expr)
        elif isinstance(inner, And):
            return Or(move_negation_inwards(Not(inner.left)), move_negation_inwards(Not(inner.right)))
        elif isinstance(inner, Or):
            return And(move_negation_inwards(Not(inner.left)), move_negation_inwards(Not(inner.right)))
        elif isinstance(inner, ForAll):
            return Exists(inner.var, move_negation_inwards(Not(inner.expr)))
        elif isinstance(inner, Exists):
            return ForAll(inner.var, move_negation_inwards(Not(inner.expr)))
        return Not(move_negation_inwards(inner))
    elif isinstance(expr, (And, Or)):
        return type(expr)(move_negation_inwards(expr.left), move_negation_inwards(expr.right))
    elif isinstance(expr, (ForAll, Exists)):
        return type(expr)(expr.var, move_negation_inwards(expr.expr))
    return expr

def substitute(expr, mapping):
    if isinstance(expr, Var):
        return mapping.get(expr.name, expr)
    elif isinstance(expr, (Func, Pred)):
        return type(expr)(expr.name, [substitute(arg, mapping) for arg in expr.args])
    elif isinstance(expr, Not):
        return Not(substitute(expr.expr, mapping))
    elif isinstance(expr, (And, Or)):
        return type(expr)(substitute(expr.left, mapping), substitute(expr.right, mapping))
    elif isinstance(expr, (ForAll, Exists)):
        return type(expr)(mapping.get(expr.var.name, expr.var), substitute(expr.expr, mapping))
    return expr

def standardize_variables(expr, mapping=None):
    if mapping is None:
        mapping = {}
    if isinstance(expr, (ForAll, Exists)):
        new_v = fresh_var(expr.var.name)
        new_map = mapping.copy()
        new_map[expr.var.name] = new_v
        return type(expr)(new_v, standardize_variables(expr.expr, new_map))
    elif isinstance(expr, (And, Or)):
        return type(expr)(standardize_variables(expr.left, mapping), standardize_variables(expr.right, mapping))
    elif isinstance(expr, Not):
        return Not(standardize_variables(expr.expr, mapping))
    elif isinstance(expr, Pred):
        return Pred(expr.name, [substitute(arg, mapping) for arg in expr.args])
    return expr

def skolemize(expr, univ_vars=None):
    if univ_vars is None:
        univ_vars = []
    if isinstance(expr, ForAll):
        return ForAll(expr.var, skolemize(expr.expr, univ_vars + [expr.var]))
    elif isinstance(expr, Exists):
        sk_term = fresh_skolem(univ_vars)
        return skolemize(substitute(expr.expr, {expr.var.name: sk_term}), univ_vars)
    elif isinstance(expr, (And, Or)):
        return type(expr)(skolemize(expr.left, univ_vars), skolemize(expr.right, univ_vars))
    elif isinstance(expr, Not):
        return Not(skolemize(expr.expr, univ_vars))
    return expr

def drop_universal(expr):
    if isinstance(expr, ForAll):
        return drop_universal(expr.expr)
    elif isinstance(expr, (And, Or)):
        return type(expr)(drop_universal(expr.left), drop_universal(expr.right))
    elif isinstance(expr, Not):
        return Not(drop_universal(expr.expr))
    return expr

def distribute_or(expr):
    if isinstance(expr, Or):
        A = distribute_or(expr.left)
        B = distribute_or(expr.right)
        if isinstance(A, And):
            return And(distribute_or(Or(A.left, B)), distribute_or(Or(A.right, B)))
        if isinstance(B, And):
            return And(distribute_or(Or(A, B.left)), distribute_or(Or(A, B.right)))
        return Or(A, B)
    elif isinstance(expr, And):
        return And(distribute_or(expr.left), distribute_or(expr.right))
    return expr

def extract_clauses(expr):
    if isinstance(expr, And):
        return extract_clauses(expr.left) + extract_clauses(expr.right)
    elif isinstance(expr, Or):
        return [extract_literals(expr)]
    return [[expr]]

def extract_literals(expr):
    if isinstance(expr, Or):
        return extract_literals(expr.left) + extract_literals(expr.right)
    return [expr]

def convert_to_cnf(formula_str, label="", verbose=True):
    expr = parse_fol(formula_str)
    s1 = eliminate_implications(expr)
    s2 = move_negation_inwards(s1)
    s3 = standardize_variables(s2)
    s4 = skolemize(s3)
    s5 = drop_universal(s4)
    s6 = distribute_or(s5)
    if verbose:
        print(f"\n--- {label} ---")
        print(f"Parsed FOL:      {expr}")
        print(f"1. Elim Implies: {s1}")
        print(f"2. Neg Inwards:  {s2}")
        print(f"3. Standardize:  {s3}")
        print(f"4. Skolemize:    {s4}")
        print(f"5. Drop ForAll:  {s5}")
        print(f"6. CNF Form:     {s6}")
    return extract_clauses(s6)

# ==========================================
# 4. UNIFICATION & RESOLUTION ENGINE
# ==========================================

def occur_check(var, term, subst):
    if var == term: return True
    if isinstance(term, Var) and term.name in subst:
        return occur_check(var, subst[term.name], subst)
    if isinstance(term, Func):
        return any(occur_check(var, arg, subst) for arg in term.args)
    return False

def unify(x, y, subst=None):
    if subst is None: subst = {}
    if x == y: return subst
    if isinstance(x, Var): return unify_var(x, y, subst)
    if isinstance(y, Var): return unify_var(y, x, subst)
    if isinstance(x, (Pred, Func)) and isinstance(y, (Pred, Func)):
        if x.name != y.name or len(x.args) != len(y.args): return None
        for a, b in zip(x.args, y.args):
            subst = unify(a, b, subst)
            if subst is None: return None
        return subst
    return None

def unify_var(var, x, subst):
    if var.name in subst: return unify(subst[var.name], x, subst)
    if isinstance(x, Var) and x.name in subst: return unify(var, subst[x.name], subst)
    if occur_check(var, x, subst): return None
    new_subst = subst.copy()
    new_subst[var.name] = x
    return new_subst

def apply_subst_literal(lit, subst):
    if isinstance(lit, Not):
        return Not(Pred(lit.expr.name, [substitute(a, subst) for a in lit.expr.args]))
    return Pred(lit.name, [substitute(a, subst) for a in lit.args])

def standardize_clause(clause):
    mapping = {}
    def rename(t):
        if isinstance(t, Var):
            if t.name not in mapping: mapping[t.name] = fresh_var("v")
            return mapping[t.name]
        if isinstance(t, Func):
            return Func(t.name, [rename(a) for a in t.args])
        return t
    res = []
    for lit in clause:
        if isinstance(lit, Not):
            res.append(Not(Pred(lit.expr.name, [rename(a) for a in lit.expr.args])))
        else:
            res.append(Pred(lit.name, [rename(a) for a in lit.args]))
    return res

def resolve_clauses(c1, c2):
    c1_std = standardize_clause(c1)
    c2_std = standardize_clause(c2)
    resolvents = []
    for i, l1 in enumerate(c1_std):
        for j, l2 in enumerate(c2_std):
            if isinstance(l1, Not) and not isinstance(l2, Not) and l1.expr.name == l2.name:
                subst = unify(l1.expr, l2)
            elif not isinstance(l1, Not) and isinstance(l2, Not) and l1.name == l2.expr.name:
                subst = unify(l1, l2.expr)
            else:
                continue
            if subst is not None:
                rem1 = [apply_subst_literal(lit, subst) for k, lit in enumerate(c1_std) if k != i]
                rem2 = [apply_subst_literal(lit, subst) for k, lit in enumerate(c2_std) if k != j]
                combined = []
                for lit in rem1 + rem2:
                    if lit not in combined:
                        combined.append(lit)
                resolvents.append((combined, subst))
    return resolvents

# ==========================================
# 5. DYNAMIC PROOF TREE VISUALIZER
# ==========================================

def plot_dynamic_proof_tree(nodes, goal_idx, title="Dynamic Resolution Proof Tree"):
    children_map = {}
    depths = {}
    def compute_depth(idx):
        if idx in depths:
            return depths[idx]
        _, p1, p2, _ = nodes[idx]
        if p1 is None:
            depths[idx] = 0
        else:
            children_map[idx] = (p1, p2)
            depths[idx] = 1 + max(compute_depth(p1), compute_depth(p2))
        return depths[idx]
    max_depth = compute_depth(goal_idx)

    coords = {}
    leaf_counter = [0]
    def assign_coords(idx):
        _, p1, p2, _ = nodes[idx]
        y = max_depth - depths[idx]
        if p1 is None:
            x = leaf_counter[0] * 3.2
            leaf_counter[0] += 1
        else:
            assign_coords(p1)
            assign_coords(p2)
            x = (coords[p1][0] + coords[p2][0]) / 2.0
        coords[idx] = (x, y)
    assign_coords(goal_idx)

    fig_w = max(10, leaf_counter[0] * 2.8)
    fig_h = max(6, (max_depth + 1) * 1.6)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=140)
    ax.axis('off')
    ax.set_title(title, fontsize=14, fontweight='bold', pad=15)
    for idx, (x, y) in coords.items():
        clause, p1, p2, subst = nodes[idx]
        label = " ∨ ".join(map(str, clause)) if clause else "{ }  Hence Proved."
        if not clause:
            fc, ec, fw = '#f0fff4', '#2f855a', 'bold'
        elif p1 is None:
            fc, ec, fw = '#fffaf0', '#dd6b20', 'normal'
        else:
            fc, ec, fw = '#ebf8ff', '#2b6cb0', 'normal'
        ax.text(x, y, label, ha='center', va='center', fontsize=9.5, fontweight=fw,
                bbox=dict(boxstyle="round,pad=0.4", fc=fc, ec=ec, lw=1.5), zorder=3)
        if p1 is not None:
            x1, y1 = coords[p1]
            x2, y2 = coords[p2]
            ax.plot([x1, x], [y1 - 0.15, y + 0.15], color='#4a5568', lw=1.3, zorder=1)
            ax.plot([x2, x], [y2 - 0.15, y + 0.15], color='#4a5568', lw=1.3, zorder=1)
            if subst:
                subst_txt = "{" + ", ".join(f"{v}/{k}" for k, v in subst.items()) + "}"
                ax.text(x + 0.25, (y1 + y) / 2.0, subst_txt, fontsize=8.5, color='#c53030', fontweight='bold')
    xs = [c[0] for c in coords.values()]
    ys = [c[1] for c in coords.values()]
    ax.set_xlim(min(xs) - 2.5, max(xs) + 2.5)
    ax.set_ylim(min(ys) - 0.6, max(ys) + 0.6)
    plt.tight_layout()
    plt.show()

# ==========================================
# 6. GENERIC RESOLUTION SOLVER DRIVER
# ==========================================

def solve_resolution(kb_strings, goal_string, verbose_cnf=True):
    global var_counter, skolem_counter
    var_counter = 0
    skolem_counter = 0
    print("==========================================")
    print("STEP 1: PARSE & CONVERT TO CNF")
    print("==========================================")
    nodes = []
    for i, s in enumerate(kb_strings):
        for c in convert_to_cnf(s, label=f"KB[{i+1}]", verbose=verbose_cnf):
            nodes.append((c, None, None, None))
    neg_goal_str = f"~({goal_string})"
    for c in convert_to_cnf(neg_goal_str, label="Negated Goal", verbose=verbose_cnf):
        nodes.append((c, None, None, None))
    print("\n==========================================")
    print("INITIAL CLAUSE SET")
    print("==========================================")
    for idx, (c, _, _, _) in enumerate(nodes):
        print(f"C{idx+1}: {' ∨ '.join(map(str, c))}")
    print("\n==========================================")
    print("STEP 2: RESOLUTION TRACE")
    print("==========================================")
    seen = {tuple(sorted(map(str, c))) for c, _, _, _ in nodes}
    i = 0
    while i < len(nodes):
        for j in range(i):
            resolvents = resolve_clauses(nodes[i][0], nodes[j][0])
            for res, subst in resolvents:
                step_num = len(nodes) + 1
                subst_str = f" using {subst}" if subst else ""
                if not res:
                    print(f"C{step_num}: {{ }} (EMPTY CLAUSE) <-- Resolved C{i+1} & C{j+1}{subst_str}")
                    print("\n>>> { } HENCE PROVED! <<<")
                    nodes.append(([], i, j, subst))
                    plot_dynamic_proof_tree(nodes, len(nodes) - 1, title=f"Proof Tree: {goal_string}")
                    return True
                sig = tuple(sorted(map(str, res)))
                if sig not in seen:
                    seen.add(sig)
                    print(f"C{step_num}: {' ∨ '.join(map(str, res))} <-- Resolved C{i+1} & C{j+1}{subst_str}")
                    nodes.append((res, i, j, subst))
        i += 1
    print("\n>>> COULD NOT PROVE (Empty clause {} cannot be derived) <<<")
    return False

# ==========================================
# 7. INTERACTIVE USER INPUT RUNNER
# ==========================================

if __name__ == "__main__":
    print("==========================================")
    print("GENERIC FOL CNF & RESOLUTION SOLVER")
    print("==========================================")
    print("Supported symbols:")
    print("  Quantifiers : ∀x  or  forall x  or  (∀x)")
    print("                ∃y  or  exists y  or  (∃y)")
    print("  Operators   : -> / → (implies), <-> / ↔ (iff)")
    print("                ~ / ¬ (not), & / ∧ (and), | / ∨ (or)")
    print("------------------------------------------")
    print("Select Mode:")
    print("  1. Proof by Resolution (Enter KB premises + Goal + Plot Graph)")
    print("  2. Convert a single FOL formula to CNF (Step-by-step)")
    choice = input("\nEnter choice (1 or 2): ").strip()
    if choice == "2":
        formula = input("\nEnter FOL formula: ").strip()
        clauses = convert_to_cnf(formula, label="User Formula", verbose=True)
        print("\nFinal Extracted Clauses:")
        for idx, c in enumerate(clauses):
            print(f"  C{idx+1}: {' ∨ '.join(map(str, c))}")
    else:
        print("\nEnter Knowledge Base (KB) premises one by one.")
        print("Type 'done' (or press Enter on a blank line) when finished:\n")
        kb_user = []
        while True:
            premise = input(f"Premise {len(kb_user)+1}: ").strip()
            if not premise or premise.lower() == "done":
                break
            kb_user.append(premise)
        if not kb_user:
            print("No premises entered. Exiting.")
        else:
            goal_user = input("\nEnter Goal statement to prove: ").strip()
            solve_resolution(kb_user, goal_user, verbose_cnf=True)

GENERIC FOL CNF & RESOLUTION SOLVER
Supported symbols:
  Quantifiers : ∀x  or  forall x  or  (∀x)
                ∃y  or  exists y  or  (∃y)
  Operators   : -> / → (implies), <-> / ↔ (iff)
                ~ / ¬ (not), & / ∧ (and), | / ∨ (or)
------------------------------------------
Select Mode:
  1. Proof by Resolution (Enter KB premises + Goal + Plot Graph)
  2. Convert a single FOL formula to CNF (Step-by-step)

Enter choice (1 or 2): 1

Enter Knowledge Base (KB) premises one by one.
Type 'done' (or press Enter on a blank line) when finished:

Premise 1: John likes all kinds of food
Premise 2: Apple and vegetables are food
Premise 3: Anything anyone eats and not killed is food.
Premise 4: Anil eats peanuts and still alive
Premise 5: Harry eats everything that Anil eats.
Premise 6: Anyone who is alive implies not killed
Premise 7: Anyone who is not killed implies alive
Premise 8: done

Enter Goal statement to prove: John likes peanuts.
STEP 1: PARSE & CONVERT TO CNF

--- KB[1] ---
P

SyntaxError: Unexpected character at position 43: . (<string>)

In [ ]:
import re
import matplotlib.pyplot as plt

# ==========================================
# 1. GENERIC ENGLISH TO FOL TRANSLATOR
# ==========================================

def norm_const(word):
    return word.strip().lower().replace(' ', '_')

def english_to_fol(sentence):
    s = sentence.strip().rstrip('.').strip()

    # 1. "[Subject] likes/loves/etc. all kind(s) of [category]"
    m = re.match(r'^([a-zA-Z]+)\s+([a-zA-Z]+)s\s+all\s+kinds?\s+of\s+([a-zA-Z_]+)$', s, re.I)
    if m:
        subj, verb, cat = norm_const(m.group(1)), m.group(2).lower(), norm_const(m.group(3))
        return f"∀x: {cat}(x) -> {verb}({subj}, x)"

    # 2. "[A] and [B] are [category]"
    m = re.match(r'^([a-zA-Z]+)\s+and\s+([a-zA-Z]+)\s+are\s+([a-zA-Z_]+)$', s, re.I)
    if m:
        a, b, cat = norm_const(m.group(1)), norm_const(m.group(2)), norm_const(m.group(3))
        return f"{cat}({a}) ∧ {cat}({b})"

    # 3. "Anything anyone [verb]s and not [state] is [category]"
    m = re.match(r'^anything\s+anyone\s+([a-zA-Z]+)s\s+and\s+not\s+([a-zA-Z]+)\s+is\s+([a-zA-Z_]+)$', s, re.I)
    if m:
        verb, state, cat = m.group(1).lower(), m.group(2).lower(), norm_const(m.group(3))
        return f"∀x ∀y: ({verb}(x, y) ∧ ¬{state}(x)) -> {cat}(y)"

    # 4. "[Subject] [verb]s [object] and (is/still) [state]"
    m = re.match(r'^([a-zA-Z]+)\s+([a-zA-Z]+)s\s+([a-zA-Z_]+)\s+and\s+(?:is\s+|still\s+)?([a-zA-Z_]+)$', s, re.I)
    if m:
        subj, verb, obj, state = norm_const(m.group(1)), m.group(2).lower(), norm_const(m.group(3)), m.group(4).lower()
        return f"{verb}({subj}, {obj}) ∧ {state}({subj})"

    # 5. "[Subject1] [verb]s everything that [Subject2] [verb]s"
    m = re.match(r'^([a-zA-Z]+)\s+([a-zA-Z]+)s\s+everything\s+that\s+([a-zA-Z]+)\s+([a-zA-Z]+)s$', s, re.I)
    if m:
        s1, v1, s2, v2 = norm_const(m.group(1)), m.group(2).lower(), norm_const(m.group(3)), m.group(4).lower()
        return f"∀x: {v2}({s2}, x) -> {v1}({s1}, x)"

    # 6. "Anyone/Everyone who is not [A] implies [B]"
    m = re.match(r'^(?:anyone|everyone)\s+who\s+is\s+not\s+([a-zA-Z_]+)\s+implies\s+([a-zA-Z_]+)$', s, re.I)
    if m:
        a, b = m.group(1).lower(), m.group(2).lower()
        return f"∀x: ¬{a}(x) -> {b}(x)"

    # 7. "Anyone/Everyone who is [A] implies not [B]"
    m = re.match(r'^(?:anyone|everyone)\s+who\s+is\s+([a-zA-Z_]+)\s+implies\s+not\s+([a-zA-Z_]+)$', s, re.I)
    if m:
        a, b = m.group(1).lower(), m.group(2).lower()
        return f"∀x: {a}(x) -> ¬{b}(x)"

    # 8. "Anyone/Everyone who [verb]s [A] also [verb]s [B] or [C]"
    m = re.match(r'^(?:anyone|everyone)\s+who\s+([a-zA-Z]+)s\s+([a-zA-Z_]+)\s+also\s+([a-zA-Z]+)s\s+([a-zA-Z_]+)\s+or\s+([a-zA-Z_]+)$', s, re.I)
    if m:
        v, a, b, c = m.group(1).lower(), norm_const(m.group(2)), norm_const(m.group(4)), norm_const(m.group(5))
        return f"∀x: {v}(x, {a}) -> ({v}(x, {b}) ∨ {v}(x, {c}))"

    # 9. "Anyone/Everyone who [verb]s [A] also [verb]s [B]"
    m = re.match(r'^(?:anyone|everyone)\s+who\s+([a-zA-Z]+)s\s+([a-zA-Z_]+)\s+also\s+([a-zA-Z]+)s\s+([a-zA-Z_]+)$', s, re.I)
    if m:
        v, a, b = m.group(1).lower(), norm_const(m.group(2)), norm_const(m.group(4))
        return f"∀x: {v}(x, {a}) -> {v}(x, {b})"

    # 10. "If someone [verb]s [A], then they [verb] [B]"
    m = re.match(r'^if\s+someone\s+([a-zA-Z]+)s\s+([a-zA-Z_]+),?\s+then\s+they\s+([a-zA-Z]+)\s+([a-zA-Z_\s]+)$', s, re.I)
    if m:
        v1, a, v2, b = m.group(1).lower(), norm_const(m.group(2)), m.group(3).lower(), norm_const(m.group(4))
        return f"∀x: {v1}(x, {a}) -> {v2}(x, {b})"

    # 11. General "All [A] are [B]" / "Every [A] is [B]"
    m = re.match(r'^(?:all|every)\s+([a-zA-Z_]+?)(?:s)?\s+(?:are|is)\s+([a-zA-Z_]+?)(?:s)?$', s, re.I)
    if m:
        a, b = m.group(1).lower(), m.group(2).lower()
        return f"∀x: {a}(x) -> {b}(x)"

    # 12. General "[Subject] is a/an [Category]" or "[Subject] is [State]"
    m = re.match(r'^([a-zA-Z]+)\s+is\s+(?:a\s+|an\
s+)?([a-zA-Z_]+)$', s, re.I)
    if m:
        subj, pred = norm_const(m.group(1)), m.group(2).lower()
        return f"{pred}({subj})"

    # 13. General binary fact "[Subject] [verb]s [Object]"
    m = re.match(r'^([a-zA-Z]+)\s+([a-zA-Z]+)s\s+([a-zA-Z_\s]+)$', s, re.I)
    if m:
        subj, verb, obj = norm_const(m.group(1)), m.group(2).lower(), norm_const(m.group(3))
        return f"{verb}({subj}, {obj})"

    return s

# ==========================================
# 2. FOL AST & PARSER
# ==========================================

class Expr: pass

class Var(Expr):
    def __init__(self, name): self.name = name
    def __repr__(self): return self.name
    def __eq__(self, o): return isinstance(o, Var) and self.name == o.name
    def __hash__(self): return hash(('Var', self.name))

class Const(Expr):
    def __init__(self, name): self.name = name
    def __repr__(self): return self.name
    def __eq__(self, o): return isinstance(o, Const) and self.name == o.name
    def __hash__(self): return hash(('Const', self.name))

class Func(Expr):
    def __init__(self, name, args): self.name, self.args = name, args
    def __repr__(self): return f"{self.name}({', '.join(map(str, self.args))})"
    def __eq__(self, o): return isinstance(o, Func) and self.name == o.name and self.args == o.args
    def __hash__(self): return hash(('Func', self.name, tuple(self.args)))

class Pred(Expr):
    def __init__(self, name, args): self.name, self.args = name, args
    def __repr__(self): return f"{self.name}({', '.join(map(str, self.args))})"
    def __eq__(self, o): return isinstance(o, Pred) and self.name == o.name and self.args == o.args
    def __hash__(self): return hash(('Pred', self.name, tuple(self.args)))

class Not(Expr):
    def __init__(self, expr): self.expr = expr
    def __repr__(self): return f"¬{self.expr}"
    def __eq__(self, o): return isinstance(o, Not) and self.expr == o.expr
    def __hash__(self): return hash(('Not', self.expr))

class And(Expr):
    def __init__(self, left, right): self.left, self.right = left, right
    def __repr__(self): return f"({self.left} ∧ {self.right})"

class Or(Expr):
    def __init__(self, left, right): self.left, self.right = left, right
    def __repr__(self): return f"({self.left} ∨ {self.right})"

class Implies(Expr):
    def __init__(self, left, right): self.left, self.right = left, right
    def __repr__(self): return f"({self.left} → {self.right})"

class ForAll(Expr):
    def __init__(self, var, expr): self.var, self.expr = var, expr
    def __repr__(self): return f"∀{self.var} {self.expr}"

class Exists(Expr):
    def __init__(self, var, expr): self.var, self.expr = var, expr
    def __repr__(self): return f"∃{self.var} {self.expr}"

TOKEN_REGEX = re.compile(
    r'\s*(?:(forall|exists|∀|∃|->|→|<->|↔|~|¬|&|∧|\||∨|\(|\)|\[|\]|,|:)|([A-Za-z_][A-Za-z0-9_]*))'
)

def tokenize(text):
    tokens, pos = [], 0
    while pos < len(text):
        m = TOKEN_REGEX.match(text, pos)
        if not m:
            raise SyntaxError(f"Could not parse text near: '{text[pos:]}'")
        pos = m.end()
        tok = m.group(1) or m.group(2)
        if tok != ':':
            tokens.append(tok)
    return tokens

class FOLParser:
    def __init__(self, tokens):
        self.tokens, self.pos = tokens, 0

    def peek(self, offset=0):
        idx = self.pos + offset
        return self.tokens[idx] if idx < len(self.tokens) else None

    def consume(self, expected=None):
        tok = self.peek()
        if expected and tok != expected:
            raise SyntaxError(f"Expected '{expected}', got '{tok}'")
        self.pos += 1
        return tok

    def parse(self, bound=None):
        return self.parse_quantified(bound or set())

    def parse_quantified(self, bound):
        tok = self.peek()
        if tok in ('(', '[') and self.peek(1) in ('forall', 'exists', '∀', '∃'):
            close_b = ')' if tok == '(' else ']'
            self.consume()
            q, v = self.consume(), self.consume()
            self.consume(close_b)
            body = self.parse_quantified(bound | {v})
            return ForAll(Var(v), body) if q in ('forall', '∀') else Exists(Var(v), body)
        if tok in ('forall', 'exists', '∀', '∃'):
            q, v = self.consume(), self.consume()
            body = self.parse_quantified(bound | {v})
            return ForAll(Var(v), body) if q in ('forall', '∀') else Exists(Var(v), body)
        return self.parse_iff(bound)

    def parse_iff(self, bound):
        left = self.parse_implies(bound)
        while self.peek() in ('<->', '↔'):
            self.consume()
            right = self.parse_implies(bound)
            left = And(Implies(left, right), Implies(right, left))
        return left

    def parse_implies(self, bound):
        left = self.parse_or(bound)
        if self.peek() in ('->', '→'):
            self.consume()
            return Implies(left, self.parse_implies(bound))
        return left

    def parse_or(self, bound):
        left = self.parse_and(bound)
        while self.peek() in ('|', '∨'):
            self.consume()
            left = Or(left, self.parse_and(bound))
        return left

    def parse_and(self, bound):
        left = self.parse_unary(bound)
        while self.peek() in ('&', '∧'):
            self.consume()
            left = And(left, self.parse_unary(bound))
        return left

    def parse_unary(self, bound):
        tok = self.peek()
        if tok in ('~', '¬'):
            self.consume()
            return Not(self.parse_unary(bound))
        if tok in ('(', '['):
            close_b = ')' if tok == '(' else ']'
            self.consume()
            expr = self.parse_quantified(bound)
            self.consume(close_b)
            return expr
        return self.parse_predicate(bound)

    def parse_predicate(self, bound):
        name = self.consume()
        args = []
        if self.peek() == '(':
            self.consume('(')
            if self.peek() != ')':
                args.append(self.parse_term(bound))
                while self.peek() == ',':
                    self.consume(',')
                    args.append(self.parse_term(bound))
            self.consume(')')
        return Pred(name, args)

    def parse_term(self, bound):
        name = self.consume()
        if self.peek() == '(':
            self.consume('(')
            args = []
            if self.peek() != ')':
                args.append(self.parse_term(bound))
                while self.peek() == ',':
                    self.consume(',')
                    args.append(self.parse_term(bound))
            self.consume(')')
            return Func(name, args)
        return Var(name) if name in bound else Const(name)

def parse_fol(formula_str):
    return FOLParser(tokenize(formula_str)).parse()

# ==========================================
# 3. CNF PIPELINE
# ==========================================

var_counter = 0
skolem_counter = 0

def fresh_var(base="v"):
    global var_counter
    var_counter += 1
    return Var(f"{base}{var_counter}")

def fresh_skolem(univ_vars):
    global skolem_counter
    skolem_counter += 1
    if not univ_vars:
        return Const(f"SkC{skolem_counter}")
    return Func(f"f{skolem_counter}", list(univ_vars))

def eliminate_implications(expr):
    if isinstance(expr, Implies):
        return Or(Not(eliminate_implications(expr.left)), eliminate_implications(expr.right))
    elif isinstance(expr, Not):
        return Not(eliminate_implications(expr.expr))
    elif isinstance(expr, (And, Or)):
        return type(expr)(eliminate_implications(expr.left), eliminate_implications(expr.right))
    elif isinstance(expr, (ForAll, Exists)):
        return type(expr)(expr.var, eliminate_implications(expr.expr))
    return expr

def move_negation_inwards(expr):
    if isinstance(expr, Not):
        inner = expr.expr
        if isinstance(inner, Not):
            return move_negation_inwards(inner.expr)
        elif isinstance(inner, And):
            return Or(move_negation_inwards(Not(inner.left)), move_negation_inwards(Not(inner.right)))
        elif isinstance(inner, Or):
            return And(move_negation_inwards(Not(inner.left)), move_negation_inwards(Not(inner.right)))
        elif isinstance(inner, ForAll):
            return Exists(inner.var, move_negation_inwards(Not(inner.expr)))
        elif isinstance(inner, Exists):
            return ForAll(inner.var, move_negation_inwards(Not(inner.expr)))
        return Not(move_negation_inwards(inner))
    elif isinstance(expr, (And, Or)):
        return type(expr)(move_negation_inwards(expr.left), move_negation_inwards(expr.right))
    elif isinstance(expr, (ForAll, Exists)):
        return type(expr)(expr.var, move_negation_inwards(expr.expr))
    return expr

def substitute(expr, mapping):
    if isinstance(expr, Var):
        return mapping.get(expr.name, expr)
    elif isinstance(expr, (Func, Pred)):
        return type(expr)(expr.name, [substitute(arg, mapping) for arg in expr.args])
    elif isinstance(expr, Not):
        return Not(substitute(expr.expr, mapping))
    elif isinstance(expr, (And, Or)):
        return type(expr)(substitute(expr.left, mapping), substitute(expr.right, mapping))
    elif isinstance(expr, (ForAll, Exists)):
        return type(expr)(mapping.get(expr.var.name, expr.var), substitute(expr.expr, mapping))
    return expr

def standardize_variables(expr, mapping=None):
    if mapping is None:
        mapping = {}
    if isinstance(expr, (ForAll, Exists)):
        new_v = fresh_var(expr.var.name)
        new_map = mapping.copy()
        new_map[expr.var.name] = new_v
        return type(expr)(new_v, standardize_variables(expr.expr, new_map))
    elif isinstance(expr, (And, Or)):
        return type(expr)(standardize_variables(expr.left, mapping), standardize_variables(expr.right, mapping))
    elif isinstance(expr, Not):
        return Not(standardize_variables(expr.expr, mapping))
    elif isinstance(expr, Pred):
        return Pred(expr.name, [substitute(arg, mapping) for arg in expr.args])
    return expr

def skolemize(expr, univ_vars=None):
    if univ_vars is None:
        univ_vars = []
    if isinstance(expr, ForAll):
        return ForAll(expr.var, skolemize(expr.expr, univ_vars + [expr.var]))
    elif isinstance(expr, Exists):
        sk_term = fresh_skolem(univ_vars)
        return skolemize(substitute(expr.expr, {expr.var.name: sk_term}), univ_vars)
    elif isinstance(expr, (And, Or)):
        return type(expr)(skolemize(expr.left, univ_vars), skolemize(expr.right, univ_vars))
    elif isinstance(expr, Not):
        return Not(skolemize(expr.expr, univ_vars))
    return expr

def drop_universal(expr):
    if isinstance(expr, ForAll):
        return drop_universal(expr.expr)
    elif isinstance(expr, (And, Or)):
        return type(expr)(drop_universal(expr.left), drop_universal(expr.right))
    elif isinstance(expr, Not):
        return Not(drop_universal(expr.expr))
    return expr

def distribute_or(expr):
    if isinstance(expr, Or):
        A = distribute_or(expr.left)
        B = distribute_or(expr.right)
        if isinstance(A, And):
            return And(distribute_or(Or(A.left, B)), distribute_or(Or(A.right, B)))
        if isinstance(B, And):
            return And(distribute_or(Or(A, B.left)), distribute_or(Or(A, B.right)))
        return Or(A, B)
    elif isinstance(expr, And):
        return And(distribute_or(expr.left), distribute_or(expr.right))
    return expr

def extract_clauses(expr):
    if isinstance(expr, And):
        return extract_clauses(expr.left) + extract_clauses(expr.right)
    elif isinstance(expr, Or):
        return [extract_literals(expr)]
    return [[expr]]

def extract_literals(expr):
    if isinstance(expr, Or):
        return extract_literals(expr.left) + extract_literals(expr.right)
    return [expr]

def convert_to_cnf(formula_str, label="", verbose=True):
    translated = english_to_fol(formula_str)
    expr = parse_fol(translated)
    s1 = eliminate_implications(expr)
    s2 = move_negation_inwards(s1)
    s3 = standardize_variables(s2)
    s4 = skolemize(s3)
    s5 = drop_universal(s4)
    s6 = distribute_or(s5)
    if verbose:
        print(f"\n--- {label} ---")
        print(f"Raw English:     {formula_str}")
        print(f"Parsed FOL:      {expr}")
        print(f"CNF Form:        {s6}")
    return extract_clauses(s6)

# ==========================================
# 4. UNIFICATION & RESOLUTION ENGINE
# ==========================================

def occur_check(var, term, subst):
    if var == term: return True
    if isinstance(term, Var) and term.name in subst:
        return occur_check(var, subst[term.name], subst)
    if isinstance(term, Func):
        return any(occur_check(var, arg, subst) for arg in term.args)
    return False

def unify(x, y, subst=None):
    if subst is None: subst = {}
    if x == y: return subst
    if isinstance(x, Var): return unify_var(x, y, subst)
    if isinstance(y, Var): return unify_var(y, x, subst)
    if isinstance(x, (Pred, Func)) and isinstance(y, (Pred, Func)):
        if x.name != y.name or len(x.args) != len(y.args): return None
        for a, b in zip(x.args, y.args):
            subst = unify(a, b, subst)
            if subst is None: return None
        return subst
    return None

def unify_var(var, x, subst):
    if var.name in subst: return unify(subst[var.name], x, subst)
    if isinstance(x, Var) and x.name in subst: return unify(var, subst[x.name], subst)
    if occur_check(var, x, subst): return None
    new_subst = subst.copy()
    new_subst[var.name] = x
    return new_subst

def apply_subst_literal(lit, subst):
    if isinstance(lit, Not):
        return Not(Pred(lit.expr.name, [substitute(a, subst) for a in lit.expr.args]))
    return Pred(lit.name, [substitute(a, subst) for a in lit.args])

def standardize_clause(clause):
    mapping = {}
    def rename(t):
        if isinstance(t, Var):
            if t.name not in mapping: mapping[t.name] = fresh_var("v")
            return mapping[t.name]
        if isinstance(t, Func):
            return Func(t.name, [rename(a) for a in t.args])
        return t
    res = []
    for lit in clause:
        if isinstance(lit, Not):
            res.append(Not(Pred(lit.expr.name, [rename(a) for a in lit.expr.args])))
        else:
            res.append(Pred(lit.name, [rename(a) for a in lit.args]))
    return res

def resolve_clauses(c1, c2):
    c1_std = standardize_clause(c1)
    c2_std = standardize_clause(c2)
    resolvents = []
    for i, l1 in enumerate(c1_std):
        for j, l2 in enumerate(c2_std):
            if isinstance(l1, Not) and not isinstance(l2, Not) and l1.expr.name == l2.name:
                subst = unify(l1.expr, l2)
            elif not isinstance(l1, Not) and isinstance(l2, Not) and l1.name == l2.expr.name:
                subst = unify(l1, l2.expr)
            else:
                continue
            if subst is not None:
                rem1 = [apply_subst_literal(lit, subst) for k, lit in enumerate(c1_std) if k != i]
                rem2 = [apply_subst_literal(lit, subst) for k, lit in enumerate(c2_std) if k != j]
                combined = []
                for lit in rem1 + rem2:
                    if lit not in combined:
                        combined.append(lit)
                resolvents.append((combined, subst))
    return resolvents

# ==========================================
# 5. DYNAMIC PROOF TREE VISUALIZER
# ==========================================

def plot_proof_tree(nodes, goal_idx, title="Resolution Proof Tree"):
    children_map = {}
    depths = {}
    def compute_depth(idx):
        if idx in depths: return depths[idx]
        _, p1, p2, _ = nodes[idx]
        if p1 is None:
            depths[idx] = 0
        else:
            children_map[idx] = (p1, p2)
            depths[idx] = 1 + max(compute_depth(p1), compute_depth(p2))
        return depths[idx]
    max_depth = compute_depth(goal_idx)

    coords = {}
    leaf_counter = [0]
    def assign_coords(idx):
        _, p1, p2, _ = nodes[idx]
        y = max_depth - depths[idx]
        if p1 is None:
            x = leaf_counter[0] * 3.2
            leaf_counter[0] += 1
        else:
            assign_coords(p1)
            assign_coords(p2)
            x = (coords[p1][0] + coords[p2][0]) / 2.0
        coords[idx] = (x, y)
    assign_coords(goal_idx)

    fig_w = max(10, leaf_counter[0] * 2.8)
    fig_h = max(6, (max_depth + 1) * 1.6)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=140)
    ax.axis('off')
    ax.set_title(title, fontsize=14, fontweight='bold', pad=15)
    for idx, (x, y) in coords.items():
        clause, p1, p2, subst = nodes[idx]
        label = " ∨ ".join(map(str, clause)) if clause else "{ }  Hence Proved."
        if not clause:
            fc, ec, fw = '#f0fff4', '#2f855a', 'bold'
        elif p1 is None:
            fc, ec, fw = '#fffaf0', '#dd6b20', 'normal'
        else:
            fc, ec, fw = '#ebf8ff', '#2b6cb0', 'normal'
        ax.text(x, y, label, ha='center', va='center', fontsize=9.5, fontweight=fw,
                bbox=dict(boxstyle="round,pad=0.4", fc=fc, ec=ec, lw=1.5), zorder=3)
        if p1 is not None:
            x1, y1 = coords[p1]
            x2, y2 = coords[p2]
            ax.plot([x1, x], [y1 - 0.15, y + 0.15], color='#4a5568', lw=1.3, zorder=1)
            ax.plot([x2, x], [y2 - 0.15, y + 0.15], color='#4a5568', lw=1.3, zorder=1)
            if subst:
                subst_txt = "{" + ", ".join(f"{v}/{k}" for k, v in subst.items()) + "}"
                ax.text(x + 0.25, (y1 + y) / 2.0, subst_txt, fontsize=8.5, color='#c53030', fontweight='bold')
    xs = [c[0] for c in coords.values()]
    ys = [c[1] for c in coords.values()]
    ax.set_xlim(min(xs) - 2.5, max(xs) + 2.5)
    ax.set_ylim(min(ys) - 0.6, max(ys) + 0.6)
    plt.tight_layout()
    plt.show()

# ==========================================
# 6. GENERIC RESOLUTION SOLVER DRIVER
# ==========================================

def solve_resolution(kb_strings, goal_string, verbose_cnf=True):
    global var_counter, skolem_counter
    var_counter = 0
    skolem_counter = 0
    print("==========================================")
    print("STEP 1: PARSE & CONVERT TO CNF")
    print("==========================================")
    nodes = []
    for i, s in enumerate(kb_strings):
        for c in convert_to_cnf(s, label=f"KB[{i+1}]", verbose=verbose_cnf):
            nodes.append((c, None, None, None))
    neg_goal_str = f"~({goal_string})"
    for c in convert_to_cnf(neg_goal_str, label="Negated Goal", verbose=verbose_cnf):
        nodes.append((c, None, None, None))
    print("\n==========================================")
    print("INITIAL CLAUSE SET")
    print("==========================================")
    for idx, (c, _, _, _) in enumerate(nodes):
        print(f"C{idx+1}: {' ∨ '.join(map(str, c))}")
    print("\n==========================================")
    print("STEP 2: RESOLUTION TRACE")
    print("\n>>> COULD NOT PROVE (Empty clause {} cannot be derived) <<<")

In [ ]:
def solve_resolution_fully(kb_strings, goal_string, verbose_cnf=True):
    global var_counter, skolem_counter
    var_counter = 0
    skolem_counter = 0
    print("==========================================")
    print("STEP 1: PARSE & CONVERT TO CNF")
    print("==========================================")
    nodes = []
    for i, s in enumerate(kb_strings):
        for c in convert_to_cnf(s, label=f"KB[{i+1}]", verbose=verbose_cnf):
            nodes.append((c, None, None, None))
    neg_goal_str = f"~({goal_string})"
    for c in convert_to_cnf(neg_goal_str, label="Negated Goal", verbose=verbose_cnf):
        nodes.append((c, None, None, None))
    print("\n==========================================")
    print("INITIAL CLAUSE SET")
    print("==========================================")
    for idx, (c, _, _, _) in enumerate(nodes):
        print(f"C{idx+1}: {' ∨ '.join(map(str, c))}")
    print("\n==========================================")
    print("STEP 2: RESOLUTION TRACE")
    print("==========================================")
    seen = {tuple(sorted(map(str, c))) for c, _, _, _ in nodes}
    i = 0
    while i < len(nodes):
        for j in range(i):
            resolvents = resolve_clauses(nodes[i][0], nodes[j][0])
            for res, subst in resolvents:
                step_num = len(nodes) + 1
                subst_str = f" using {subst}" if subst else ""
                if not res:
                    print(f"C{step_num}: {{ }} (EMPTY CLAUSE) <-- Resolved C{i+1} & C{j+1}{subst_str}")
                    print("\n>>> { } HENCE PROVED! <<<\n")
                    nodes.append(([], i, j, subst))
                    plot_proof_tree(nodes, len(nodes) - 1, title=f"Proof Tree: {goal_string}")
                    return True
                sig = tuple(sorted(map(str, res)))
                if sig not in seen:
                    seen.add(sig)
                    print(f"C{step_num}: {' ∨ '.join(map(str, res))} <-- Resolved C{i+1} & C{j+1}{subst_str}")
                    nodes.append((res, i, j, subst))
        i += 1
    print("\n>>> COULD NOT PROVE (Empty clause {} cannot be derived) <<<")
    return False

# Define English premises and goal
kb_english = [
    "John likes all kinds of food",
    "Apple and vegetables are food",
    "Anything anyone eats and not killed is food",
    "Anil eats peanuts and still alive",
    "Harry eats everything that Anil eats",
    "Anyone who is alive implies not killed",
    "Anyone who is not killed implies alive"
]
goal_english = "John likes peanuts"

# Run proof and plot dynamic proof tree
solve_resolution_fully(kb_english, goal_english, verbose_cnf=True)

In [ ]:
import re

# ==========================================
# 1. DATA STRUCTURES FOR FOL TERMS & CLAUSES
# ==========================================
class Literal:
    def __init__(self, predicate, args, negated=False):
        self.predicate = predicate
        self.args = tuple(args)  # Lowercase = variable, Capitalized = constant
        self.negated = negated

    def negate(self):
        return Literal(self.predicate, self.args, not self.negated)

    def substitute(self, theta):
        new_args = [theta.get(arg, arg) for arg in self.args]
        return Literal(self.predicate, new_args, self.negated)

    def is_complement(self, other):
        return self.predicate == other.predicate and self.negated != other.negated

    def __eq__(self, other):
        return (self.predicate == other.predicate and
                self.args == other.args and
                self.negated == other.negated)

    def __hash__(self):
        return hash((self.predicate, self.args, self.negated))

    def __str__(self):
        prefix = "¬ " if self.negated else ""
        return f"{prefix}{self.predicate}({', '.join(self.args)})"


class Clause:
    def __init__(self, literals, parents=None, subst=None, label=""):
        seen = set()
        unique_lits = []
        for lit in literals:
            if lit not in seen:
                seen.add(lit)
                unique_lits.append(lit)
        self.literals = tuple(unique_lits)
        self.parents = parents
        self.subst = subst or {}
        self.label = label

    def is_empty(self):
        return len(self.literals) == 0

    def __eq__(self, other):
        return set(self.literals) == set(other.literals)

    def __hash__(self):
        return hash(frozenset(self.literals))

    def __str__(self):
        if self.is_empty():
            return "{   }  Hence proved."
        return " V ".join(str(lit) for lit in self.literals)


# ==========================================
# 2. PARSER & UNIFICATION ENGINE
# ==========================================
def parse_literal(lit_str):
    lit_str = lit_str.strip()
    negated = lit_str.startswith("~") or lit_str.startswith("¬")
    if negated:
        lit_str = lit_str[1:].strip()
    match = re.match(r"([A-Za-z0-9_]+)\(([^)]+)\)", lit_str)
    if not match:
        raise ValueError(f"Invalid literal format: {lit_str}")
    pred = match.group(1)
    args = [a.strip() for a in match.group(2).split(",")]
    return Literal(pred, args, negated)

def parse_clause(clause_str, label=""):
    parts = re.split(r"\s+(?:v|V|\||OR)\s+", clause_str.strip())
    return Clause([parse_literal(p) for p in parts], label=label)

def is_variable(term):
    return term.islower()

def unify(args1, args2):
    if len(args1) != len(args2):
        return None
    theta = {}
    for x, y in zip(args1, args2):
        while x in theta:
            x = theta[x]
        while y in theta:
            y = theta[y]
        if x == y:
            continue
        elif is_variable(x):
            theta[x] = y
        elif is_variable(y):
            theta[y] = x
        else:
            return None
    return theta


# ==========================================
# 3. LINEAR RESOLUTION ENGINE & EXACT TREE PRINTER
# ==========================================
def resolve_pair(c1, c2):
    resolvents = []
    for i, lit1 in enumerate(c1.literals):
        for j, lit2 in enumerate(c2.literals):
            if lit1.is_complement(lit2):
                theta = unify(lit1.args, lit2.args)
                if theta is not None:
                    new_lits = [
                        l.substitute(theta)
                        for idx, l in enumerate(c1.literals) if idx != i
                    ] + [
                        l.substitute(theta)
                        for idx, l in enumerate(c2.literals) if idx != j
                    ]
                    resolvents.append(Clause(new_lits, parents=(c1, c2), subst=theta))
    return resolvents

def extract_linear_steps(empty_clause):
    """Extracts the top-down linear proof chain starting from the negated goal."""
    steps = []
    curr = empty_clause
    while curr.parents is not None:
        left, right = curr.parents
        # Keep the derived clause on the left, KB clause on the right
        if left.parents is None and right.parents is not None:
            left, right = right, left
        steps.append((left, right, curr.subst, curr))
        curr = left
    steps.reverse()
    return steps

def print_exact_textbook_tree(empty_clause):
    steps = extract_linear_steps(empty_clause)
    col_width = 36

    print("\n=== RESOLUTION TREE ===\n")
    for left, right, subst, result in steps:
        left_str = str(left)
        right_str = str(right)
        print(f"{left_str:<{col_width}}{right_str}")

        # Format substitution like {Peanuts/x}
        subst_str = ""
        if subst:
            pairs = [f"{val}/{var}" for var, val in subst.items()]
            subst_str = "{" + ", ".join(pairs) + "}"

        print(f"    \\         /                     {subst_str}")
        print(f"     \\       /")
        print(f"      \\     /")

    # Print final empty clause at the bottom
    final_clause = steps[-1][3]
    print(f"    {final_clause}\n")

def solve_fol_resolution(kb_items, goal_string):
    clauses = [parse_clause(s, label=label) for label, s in kb_items]
    goal_lit = parse_literal(goal_string)
    negated_goal = Clause([goal_lit.negate()], label="Negated Goal")

    # Use Linear Resolution (Set of Support starting from Negated Goal)
    # so the proof matches the exact textbook chain!
    queue = [negated_goal]
    visited = {negated_goal}

    while queue:
        current = queue.pop(0)
        for kb_clause in clauses:
            resolvents = resolve_pair(current, kb_clause)
            for res in resolvents:
                if res.is_empty():
                    print_exact_textbook_tree(res)
                    return res
                if res not in visited:
                    visited.add(res)
                    queue.append(res)

    print("Goal could not be proved.")
    return None


# ==========================================
# 4. YOUR PREMISES (a - h) LOADED & EXECUTED
# ==========================================
if __name__ == "__main__":
    kb = [
        ("a", "~food(x) v likes(John, x)"),
        ("b1", "food(Apple)"),
        ("b2", "food(Vegetables)"),
        ("c", "~eats(y, z) v killed(y) v food(z)"),
        ("d1", "eats(Anil, Peanuts)"),
        ("d2", "alive(Anil)"),
        ("e", "~eats(Anil, w) v eats(Harry, w)"),
        ("f", "killed(u) v alive(u)"),
        ("g", "~alive(k) v ~killed(k)")
    ]

    goal = "likes(John, Peanuts)"

    solve_fol_resolution(kb, goal)



=== RESOLUTION TREE ===

¬ likes(John, Peanuts)              ¬ food(x) V likes(John, x)
    \         /                     {Peanuts/x}
     \       /
      \     /
¬ food(Peanuts)                     ¬ eats(y, z) V killed(y) V food(z)
    \         /                     {Peanuts/z}
     \       /
      \     /
¬ eats(y, Peanuts) V killed(y)      eats(Anil, Peanuts)
    \         /                     {Anil/y}
     \       /
      \     /
killed(Anil)                        ¬ alive(k) V ¬ killed(k)
    \         /                     {Anil/k}
     \       /
      \     /
¬ alive(Anil)                       alive(Anil)
    \         /                     
     \       /
      \     /
    {   }  Hence proved.



In [ ]:
import re

# ==========================================
# 1. DATA STRUCTURES FOR FOL TERMS & CLAUSES
# ==========================================
class Literal:
    def __init__(self, predicate, args, negated=False):
        self.predicate = predicate
        self.args = tuple(args)  # Lowercase = variable, Capitalized = constant
        self.negated = negated

    def negate(self):
        return Literal(self.predicate, self.args, not self.negated)

    def substitute(self, theta):
        new_args = [theta.get(arg, arg) for arg in self.args]
        return Literal(self.predicate, new_args, self.negated)

    def is_complement(self, other):
        return self.predicate == other.predicate and self.negated != other.negated

    def __eq__(self, other):
        return (self.predicate == other.predicate and
                self.args == other.args and
                self.negated == other.negated)

    def __hash__(self):
        return hash((self.predicate, self.args, self.negated))

    def __str__(self):
        prefix = "¬ " if self.negated else ""
        return f"{prefix}{self.predicate}({', '.join(self.args)})"


class Clause:
    def __init__(self, literals, parents=None, subst=None, label=""):
        seen = set()
        unique_lits = []
        for lit in literals:
            if lit not in seen:
                seen.add(lit)
                unique_lits.append(lit)
        self.literals = tuple(unique_lits)
        self.parents = parents
        self.subst = subst or {}
        self.label = label

    def is_empty(self):
        return len(self.literals) == 0

    def __eq__(self, other):
        return set(self.literals) == set(other.literals)

    def __hash__(self):
        return hash(frozenset(self.literals))

    def __str__(self):
        if self.is_empty():
            return "{   }  Hence proved."
        return " V ".join(str(lit) for lit in self.literals)


# ==========================================
# 2. PARSER & UNIFICATION ENGINE
# ==========================================
def parse_literal(lit_str):
    lit_str = lit_str.strip()
    negated = lit_str.startswith("~") or lit_str.startswith("¬")
    if negated:
        lit_str = lit_str[1:].strip()
    match = re.match(r"([A-Za-z0-9_]+)\(([^)]+)\)", lit_str)
    if not match:
        raise ValueError(f"Invalid literal format: {lit_str}")
    pred = match.group(1)
    args = [a.strip() for a in match.group(2).split(",")]
    return Literal(pred, args, negated)

def parse_clause(clause_str, label=""):
    parts = re.split(r"\s+(?:v|V|\||OR)\s+", clause_str.strip())
    return Clause([parse_literal(p) for p in parts], label=label)

def is_variable(term):
    return term.islower()

def unify(args1, args2):
    if len(args1) != len(args2):
        return None
    theta = {}
    for x, y in zip(args1, args2):
        while x in theta:
            x = theta[x]
        while y in theta:
            y = theta[y]
        if x == y:
            continue
        elif is_variable(x):
            theta[x] = y
        elif is_variable(y):
            theta[y] = x
        else:
            return None
    return theta


# ==========================================
# 3. LINEAR RESOLUTION ENGINE & TREE PRINTER
# ==========================================
def resolve_pair(c1, c2):
    resolvents = []
    for i, lit1 in enumerate(c1.literals):
        for j, lit2 in enumerate(c2.literals):
            if lit1.is_complement(lit2):
                theta = unify(lit1.args, lit2.args)
                if theta is not None:
                    new_lits = [
                        l.substitute(theta)
                        for idx, l in enumerate(c1.literals) if idx != i
                    ] + [
                        l.substitute(theta)
                        for idx, l in enumerate(c2.literals) if idx != j
                    ]
                    resolvents.append(Clause(new_lits, parents=(c1, c2), subst=theta))
    return resolvents

def extract_linear_steps(target_clause):
    """Extracts the top-down linear proof chain ending at target_clause."""
    steps = []
    curr = target_clause
    while curr.parents is not None:
        left, right = curr.parents
        if left.parents is None and right.parents is not None:
            left, right = right, left
        steps.append((left, right, curr.subst, curr))
        curr = left
    steps.reverse()
    return steps

def print_exact_textbook_tree(final_clause, proved=True):
    steps = extract_linear_steps(final_clause)
    col_width = 44

    print("\n=== RESOLUTION TREE ===\n")
    for left, right, subst, result in steps:
        left_str = str(left)
        right_str = str(right)
        print(f"{left_str:<{col_width}}{right_str}")

        subst_str = ""
        if subst:
            pairs = [f"{val}/{var}" for var, val in subst.items()]
            subst_str = "{" + ", ".join(pairs) + "}"

        print(f"    \\         /                             {subst_str}")
        print(f"     \\       /")
        print(f"      \\     /")

    if proved:
        print(f"    {final_clause}\n")
    else:
        # ADD-ON: Print the residual dead-end clause when { } cannot be reached
        print(f"    {final_clause}   <-- [DEAD END: No matching clause in KB to resolve this!]")
        print("\n=== CONCLUSION ===")
        print("Null set {   } CANNOT be derived.")
        print(f"Reason: '{final_clause}' remains unresolved because Premise 1 uses OR (Physics V Mathematics).")
        print("Therefore, 'Alice likes data analysis' is NOT logically entailed by the KB.\n")

def solve_fol_resolution(kb_items, goal_string):
    clauses = [parse_clause(s, label=label) for label, s in kb_items]
    goal_lit = parse_literal(goal_string)
    negated_goal = Clause([goal_lit.negate()], label="Negated Goal")

    queue = [negated_goal]
    visited = {negated_goal}
    deepest_clause = negated_goal  # Track the furthest clause reached if proof fails

    while queue:
        current = queue.pop(0)
        deepest_clause = current
        for kb_clause in clauses:
            resolvents = resolve_pair(current, kb_clause)
            for res in resolvents:
                if res.is_empty():
                    print_exact_textbook_tree(res, proved=True)
                    return res
                if res not in visited:
                    visited.add(res)
                    queue.append(res)

    # ADD-ON TRIGGER: Empty set not found -> print the partial tree up to the dead end
    print_exact_textbook_tree(deepest_clause, proved=False)
    return None


# ==========================================
# 4. Q3 PREMISES LOADED & EXECUTED
# ==========================================
if __name__ == "__main__":
    # Note: Constants are Capitalized (Alice, Science, Physics, etc.)
    # Variables are lowercase (x, y, z)
    kb = [
        ("a", "~likes(x, Science) v likes(x, Physics) v likes(x, Mathematics)"),
        ("b", "~likes(y, Mathematics) v likes(y, Statistics)"),
        ("c", "likes(Alice, Science)"),
        ("d", "likes(Bob, Physics)"),
        ("e", "~likes(z, Statistics) v likes(z, Data_Analysis)")
    ]

    goal = "likes(Alice, Data_Analysis)"

    solve_fol_resolution(kb, goal)



=== RESOLUTION TREE ===

¬ likes(Alice, Data_Analysis)               ¬ likes(z, Statistics) V likes(z, Data_Analysis)
    \         /                             {Alice/z}
     \       /
      \     /
¬ likes(Alice, Statistics)                  ¬ likes(y, Mathematics) V likes(y, Statistics)
    \         /                             {Alice/y}
     \       /
      \     /
¬ likes(Alice, Mathematics)                 ¬ likes(x, Science) V likes(x, Physics) V likes(x, Mathematics)
    \         /                             {Alice/x}
     \       /
      \     /
¬ likes(Alice, Science) V likes(Alice, Physics)likes(Alice, Science)
    \         /                             
     \       /
      \     /
    likes(Alice, Physics)   <-- [DEAD END: No matching clause in KB to resolve this!]

=== CONCLUSION ===
Null set {   } CANNOT be derived.
Reason: 'likes(Alice, Physics)' remains unresolved because Premise 1 uses OR (Physics V Mathematics).
Therefore, 'Alice likes data analysis' is NOT lo